# Phát hiện DDoS (CIC-DDoS2019) - Deep MHA - bản Google Colab
**Cách hoạt động**
- Lần đầu: tải dataset từ Kaggle **một lần** và lưu file zip vào Google Drive (`MyDrive/KLTN_DDoS/dataset/`).
- Mỗi lần chạy: giải nén từ Drive ra ổ cứng tạm của Colab (`/content/cicddos`, đọc nhanh hơn Drive).
- Dữ liệu trung gian, dữ liệu đã xử lý, checkpoint và kết quả đều ghi vào Drive (`MyDrive/KLTN_DDoS/work/`) nên **không mất khi Colab ngắt**.
- Từ lần thứ hai, nếu `work/interim` đã có trên Drive và `REUSE_INTERIM=True` thì **không cần giải nén**, bỏ được khoảng 30-45 phút đọc CSV.

**Thứ tự chạy:** Runtime → Change runtime type → **T4 GPU** → Run all.

Bản sửa đã ghép sẵn: gộp UDPLag → UDP, lọc nhãn theo giao thức, bỏ đuôi NTP, 3b theo nhãn đa số, tokenizer v2, baseline RF + ensemble.

## BẢNG ĐIỀU KHIỂN - chọn phần nào cần chạy

In [ ]:
import os, sys, json, time, types, gc, shutil
from pathlib import Path

def _flag(name, default="1"):
    return os.environ.get(name, default) == "1"

# Mặc định: CHỈ tiền xử lý + huấn luyện Batch + baseline RF, không chạy Incremental Learning.
RUN_PREPROCESS    = True    # đọc CSV -> làm sạch -> chia theo khối thời gian -> tokenize
REUSE_INTERIM     = True    # dùng lại MyDrive/KLTN_DDoS/work/interim nếu đã có (bỏ ~30-45 phút đọc CSV).
                            # Đặt False khi đổi tham số GĐ2 (PRE_CAP_ATTACK, DEDUP_*, LABEL_CANON, DROP_SET...)
RUN_SANITY        = False   # GĐ9: bảng thống kê + RF kiểm tra rò rỉ (tạo stats_class_day_split.csv)
RUN_BATCH         = True    # huấn luyện Batch Learning trên split có sẵn
RUN_RF_BASELINE   = True    # baseline RandomForest + ensemble MHA/RF (cell sau BATCH)
RUN_DEPLOY_EXPORT = False   # hai cell XUẤT TRIỂN KHAI (tokenizer.joblib, TorchScript, bundle)
RUN_IL            = False   # Incremental Learning
RUN_A             = False   #   kịch bản A: class-incremental
RUN_B             = False   #   kịch bản B: drift theo ngày
RUN_JOINT         = False   #   kèm joint retrain (upper bound, tốn thời gian nhất)
QUICK             = _flag("QUICK", "0")   # chạy thử nhanh toàn bộ pipeline (dữ liệu rút gọn, 2 epoch)
STRICT_SPLIT      = True    # dừng NGAY nếu tỉ lệ train/val/test của một lớp lệch mạnh khỏi ~0.6/0.2/0.2

BATCH_VIEWS = ["A"]          # "A" | "all" | "cross_day"
SEEDS       = [42]           # báo cáo khóa luận: [42, 43, 44]
METHODS     = ["replay", "replay_kd", "replay_kd_feat"]

# Ghi đè cấu hình mô hình/huấn luyện (ví dụ ablation): {"per_feature_norm": True, "freeze": "last"}
CFG_OVERRIDES = {}
if QUICK:
    CFG_OVERRIDES.update(epochs=2, patience=2, il_epochs=1)
    REUSE_INTERIM = False
print(f"RUN_PREPROCESS={RUN_PREPROCESS} REUSE_INTERIM={REUSE_INTERIM} RUN_BATCH={RUN_BATCH} RF={RUN_RF_BASELINE} "
      f"EXPORT={RUN_DEPLOY_EXPORT} RUN_IL={RUN_IL} (A={RUN_A}, B={RUN_B}, joint={RUN_JOINT}) QUICK={QUICK} SEEDS={SEEDS}")

## COLAB 1 - Kiểm tra GPU, RAM, ổ đĩa

In [ ]:
import subprocess, shutil, os, psutil
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout or "KHÔNG có GPU -> Runtime > Change runtime type > T4 GPU")
print(f"RAM: {psutil.virtual_memory().total/1e9:.1f} GB | CPU: {os.cpu_count()} | "
      f"ổ /content còn trống: {shutil.disk_usage('/content').free/1e9:.0f} GB")
if psutil.virtual_memory().total < 20e9:
    print("LƯU Ý: RAM < 20 GB. Nếu GĐ3 bị tràn RAM (phiên tự khởi động lại), hạ PRE_CAP_ATTACK ở GĐ0 "
          "xuống 150_000 rồi chạy lại với REUSE_INTERIM=False, hoặc dùng Colab Pro (High-RAM).")

## COLAB 2 - Gắn Google Drive và đặt đường dẫn

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")

DRIVE_ROOT  = Path("/content/drive/MyDrive/KLTN_DDoS")     # đổi nếu muốn lưu chỗ khác
DATASET_DIR = DRIVE_ROOT / "dataset"                         # zip dataset tải từ Kaggle (tải 1 lần)
LOCAL_DATA  = Path("/content/cicddos")                       # CSV giải nén (ổ tạm của Colab, đọc nhanh)
WORK_DRIVE  = DRIVE_ROOT / "work"                            # interim / processed / results / deploy_export
for d in (DATASET_DIR, WORK_DRIVE):
    d.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET = "rodrigorosasilva/cic-ddos2019-30gb-full-dataset-csv-files"
ZIP_PATH = DATASET_DIR / (KAGGLE_DATASET.split("/")[1] + ".zip")
KEEP_DIRS = ("01-12", "03-11")                               # chỉ giải nén CSV của hai ngày dùng trong khóa luận

os.environ["INPUT_DIR"] = str(LOCAL_DATA)    # GĐ0 đọc 2 biến này
os.environ["WORK_DIR"]  = str(WORK_DRIVE)
print("Drive:", DRIVE_ROOT, "| Dữ liệu tạm:", LOCAL_DATA, "| Kết quả:", WORK_DRIVE)

## COLAB 3 - Khóa API Kaggle
Cách 1 (khuyên dùng): Colab Secrets (biểu tượng 🔑 bên trái) - thêm `KAGGLE_USERNAME` và `KAGGLE_KEY`, bật "Notebook access".
Cách 2: đặt file `kaggle.json` vào `MyDrive/KLTN_DDoS/kaggle.json`.
Chỉ cần ở lần đầu (khi chưa có zip trên Drive).

In [ ]:
import json, os, stat
from pathlib import Path

def setup_kaggle():
    kdir = Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True); kfile = kdir / "kaggle.json"
    creds = None
    try:
        from google.colab import userdata
        u, k = userdata.get("KAGGLE_USERNAME"), userdata.get("KAGGLE_KEY")
        if u and k:
            creds = {"username": u, "key": k}; print("Dùng khóa Kaggle từ Colab Secrets")
    except Exception:
        pass
    if creds is None and (DRIVE_ROOT / "kaggle.json").exists():
        creds = json.loads((DRIVE_ROOT / "kaggle.json").read_text()); print("Dùng", DRIVE_ROOT / "kaggle.json")
    if creds is None:
        return False
    kfile.write_text(json.dumps(creds)); kfile.chmod(stat.S_IRUSR | stat.S_IWUSR)
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = creds["username"], creds["key"]
    return True

HAVE_KAGGLE = setup_kaggle()
if not HAVE_KAGGLE:
    print("Chưa có khóa Kaggle (chỉ cần nếu zip dataset chưa có trên Drive).")

## COLAB 4 - Tải dataset từ Kaggle về Drive (CHỈ chạy thật ở lần đầu)

In [ ]:
import subprocess, shutil, zipfile, time

def zip_ok(p):
    try:
        with zipfile.ZipFile(p) as z:
            return any(n.lower().endswith(".csv") for n in z.namelist())
    except Exception:
        return False

if ZIP_PATH.exists() and zip_ok(ZIP_PATH):
    print(f"Đã có dataset trên Drive: {ZIP_PATH} ({ZIP_PATH.stat().st_size/1e9:.1f} GB) -> bỏ qua tải.")
else:
    assert HAVE_KAGGLE, "Cần khóa Kaggle (COLAB 3) để tải lần đầu."
    subprocess.run("pip -q install -U kaggle", shell=True, check=True)
    tmp = Path("/content/_kaggle_dl"); tmp.mkdir(exist_ok=True)
    print("Đang tải từ Kaggle về ổ tạm của Colab ..."); t0 = time.time()
    r = subprocess.run(f"kaggle datasets download -d {KAGGLE_DATASET} -p {tmp}", shell=True)
    assert r.returncode == 0, "Tải thất bại: kiểm tra khóa Kaggle / tên dataset"
    src = next(tmp.glob("*.zip"))
    gb = src.stat().st_size / 1e9
    free_drive = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
    print(f"Tải xong {gb:.1f} GB sau {time.time()-t0:.0f}s. Drive còn trống ~{free_drive:.0f} GB")
    assert free_drive > gb + 1, ("Google Drive không đủ chỗ cho file zip. Giải phóng Drive / nâng dung lượng, "
                                 "hoặc chạy tiếp mà không lưu (dữ liệu sẽ phải tải lại ở lần sau).")
    print("Đang chép zip lên Drive ..."); t0 = time.time()
    shutil.copy(src, ZIP_PATH.with_suffix(".zip.part")); ZIP_PATH.with_suffix(".zip.part").rename(ZIP_PATH)
    print(f"Đã lưu {ZIP_PATH} ({time.time()-t0:.0f}s)")
    # giữ bản tải trong ổ tạm để COLAB 5 giải nén nhanh hơn ở phiên này
    LOCAL_ZIP = src

## COLAB 5 - Nhập dataset từ Drive vào Colab cho lần chạy này

In [ ]:
import subprocess, time
INTERIM_DRIVE = WORK_DRIVE / "interim"
can_reuse = REUSE_INTERIM and INTERIM_DRIVE.exists() and any(INTERIM_DRIVE.glob("*.parquet"))
need_csv = RUN_PREPROCESS and not can_reuse

if not RUN_PREPROCESS:
    print("RUN_PREPROCESS=False -> dùng work/processed có sẵn trên Drive, không cần CSV.")
elif can_reuse:
    print(f"REUSE_INTERIM=True và đã có {len(list(INTERIM_DRIVE.glob('*.parquet')))} file trung gian trên Drive "
          "-> không cần giải nén CSV.")
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
else:
    n_csv = len(list(LOCAL_DATA.rglob("*.csv"))) if LOCAL_DATA.exists() else 0
    if n_csv >= 18:
        print(f"CSV đã được giải nén ở phiên này ({n_csv} file) -> bỏ qua.")
    else:
        zsrc = globals().get("LOCAL_ZIP", ZIP_PATH)
        assert Path(zsrc).exists(), "Chưa có dataset: chạy COLAB 3 + COLAB 4 trước."
        LOCAL_DATA.mkdir(parents=True, exist_ok=True)
        pats = " ".join(f'"*{d}/*"' for d in KEEP_DIRS)
        print(f"Giải nén {zsrc} -> {LOCAL_DATA} (chỉ {KEEP_DIRS}) ..."); t0 = time.time()
        r = subprocess.run(f'unzip -q -o "{zsrc}" {pats} -d "{LOCAL_DATA}"', shell=True)
        assert r.returncode in (0, 1), f"unzip lỗi (mã {r.returncode})"
        print(f"Xong sau {time.time()-t0:.0f}s: {len(list(LOCAL_DATA.rglob('*.csv')))} file CSV | "
              f"ổ /content còn {shutil.disk_usage('/content').free/1e9:.0f} GB")
    if "LOCAL_ZIP" in globals():                          # bản zip tải lần đầu không cần nữa
        Path(LOCAL_ZIP).unlink(missing_ok=True)

In [ ]:
# ===== GIAI ĐOẠN 0 - Cấu hình, thư viện =====
import os, re, json, gc, time
from pathlib import Path
import numpy as np
import pandas as pd

# ====== ĐƯỜNG DẪN (Colab: INPUT_DIR/WORK_DIR đặt ở cell COLAB 2; mặc định Kaggle nếu chạy ngoài Colab) ======
INPUT_DIR = Path(os.environ.get("INPUT_DIR", "/kaggle/input"))
WORK_DIR  = Path(os.environ.get("WORK_DIR",  "/kaggle/working"))
INTERIM   = WORK_DIR / "interim"      # file trung gian (mỗi ngày x nhãn x file)
PROCESSED = WORK_DIR / "processed"    # kết quả cuối để huấn luyện
for d in (INTERIM, PROCESSED):
    d.mkdir(parents=True, exist_ok=True)

# ====== THAM SỐ ======
SEED = 42
rng = np.random.default_rng(SEED)

CHUNK             = 500_000    # số dòng mỗi lần đọc CSV
DEDUP_VECTORS     = True       # True : loại các dòng có VECTOR ĐẶC TRƯNG giống hệt (sau khi đã bỏ IP/cổng/thời gian)
                               # False: giữ phân bố tự nhiên của lưu lượng (chỉ lấy mẫu đều, không loại trùng)
DROP_CONFLICTS    = True       # loại các vector giống hệt nhưng khác nhãn (mơ hồ)
MIN_SHARED_PER_DAY = 500       # kịch bản B: lớp chỉ được coi là "dùng chung hai ngày" nếu có >= ngần này mẫu MỖI ngày
DEDUP_BEFORE_SAMPLE = DEDUP_VECTORS   # nếu loại trùng thì làm TRƯỚC khi lấy mẫu
DEDUP_MODE        = "all_fields"  # "all_fields": dòng trùng = giống hệt MỌI trường của CSV (Flow ID, IP, port, Timestamp,
                                  #               đặc trưng, nhãn...; trừ cột chỉ số "Unnamed: 0") -> chỉ bỏ bản ghi lặp thật
                                  # "features"  : cách cũ - trùng vector đặc trưng trong cùng (ngày, nhãn)
INF_NAN_MODE      = "flag_impute" # "flag_impute": GIỮ dòng có inf/NaN; thêm cờ <cột>__is_posinf / __is_neginf / __is_missing,
                                  #                ±inf -> NaN -> median của tập fit (train stage 0), rồi mới tokenize
                                  # "drop"       : cách cũ - bỏ mọi dòng có inf/NaN
PURGE_EXACT_DUP   = False         # True: xóa dòng val/test TRÙNG HỆT vector đặc trưng của một dòng train cùng lớp (GĐ6)
ATTACK_WINDOWS    = {             # lọc nhãn theo cửa sổ tấn công chính thức (giờ trong cột Timestamp của CSV)
    # flow gán nhãn NTP sau 10:46 (cửa sổ chính thức 10:35-10:45) hầu hết giống DNS -> nhiễu nhãn ở đuôi cửa sổ
    ("day1", "NTP"): ("2018-12-01 10:34:00", "2018-12-01 10:46:00"),
}
PRE_CAP_ATTACK    = 300_000    # trần mỗi (file, nhãn tấn công) lúc đọc: lấy mẫu đều theo thời gian
                               # (nếu DEDUP_BEFORE_SAMPLE=True, trần này áp dụng lên các dòng DUY NHẤT)
CAP_PER_DAY_LABEL = None       # None = KHÔNG giới hạn: dùng MỌI dòng sau lọc; số nguyên = trần mỗi (ngày, nhãn tấn công)
SPLIT_FRAC        = (0.6, 0.2, 0.2)   # train / val / test theo KHỐI THỜI GIAN trong từng (ngày, nhãn)
GAP_S             = 30         # khoảng đệm (giây) giữa các khối; nên >= thời lượng flow điển hình
MAX_GAP_FRAC      = 0.02       # trần số dòng bị cắt ở MỖI phía của một ranh giới (tỉ lệ so với số dòng của lớp)
SPLIT_MODE        = "contiguous"   # "contiguous": 60% đầu = train, 20% giữa = val, 20% cuối = test (nghiêm ngặt về thời gian)
                               # "interleaved": chia thành N_BLOCKS khối liên tiếp rồi gán xen kẽ (giảm ảnh hưởng của drift trong cửa sổ tấn công)
N_BLOCKS          = 10         # chỉ dùng cho SPLIT_MODE="interleaved"
PURGE_NEAR_DUP    = False      # True: xóa val/test gần trùng train (xem giai đoạn 6)
NEAR_DEC          = 1          # số chữ số thập phân khi làm tròn để phát hiện gần-trùng
SCALER_FIT        = "A_S0"     # tokenizer/scaler chỉ fit trên train của Stage 0 kịch bản A (không rò rỉ)
                               # "day1_train": fit trên train ngày 1 (đã nhìn thấy dải giá trị của lớp tương lai)
TOKENIZER_MODE    = "ple_tokens"     # "ple_tokens": token nguyên từ tabular_tokenizer.py của bạn (cho start.py/MHA-PLE)
                               # "zscore"    : signed-log + z-score (float)     | "both": lưu cả hai
TOKEN_SCALE_FACTOR = 30_000    # = scale_factor của tabular_tokenizer.py -> NUM_BINS = 30_001
DEBUG_MAX_CHUNKS  = 1 if QUICK else None   # QUICK: chỉ đọc chunk đầu của mỗi file

BENIGN = "Benign"

# Nhận diện "ngày" từ đường dẫn file (KHÔNG dựa vào cột Timestamp vì có thể ghi ngày khác)
DAY_PATTERNS = {
    "day1": [r"01[-_]12", r"training"],   # 12/01/2019
    "day2": [r"03[-_]11", r"testing"],    # 11/03/2019
}

# Chuẩn hóa nhãn giữa hai ngày (DrDoS_DNS, UDP-lag, Syn, ...). Nhãn không có ở đây bị loại (WebDDoS, PortScan...)
LABEL_CANON = {
    "benign": "Benign", "dns": "DNS", "ldap": "LDAP", "mssql": "MSSQL", "netbios": "NetBIOS",
    "ntp": "NTP", "snmp": "SNMP", "ssdp": "SSDP", "syn": "SYN", "tftp": "TFTP",
    "udp": "UDP", "udplag": "UDPLag", "portmap": "Portmap",
}

# Cột định danh / dễ gây rò rỉ (so khớp chữ thường). Source/Destination Port bị bỏ vì reflection attack
# mang cổng dịch vụ (53, 389, 123...) trong Source Port -> "đường tắt" học nhãn.
DROP_SET = {"unnamed: 0", "flow id", "source ip", "destination ip", "source port",
            "destination port", "simillarhttp", "inbound"}

# Các stage của kịch bản A (class-incremental), theo họ tấn công
STAGES_A = [["UDP", "SYN"],                 # UDP = UDP + UDPLag (gộp ở GĐ3)
            ["DNS", "NTP", "SNMP"],
            ["LDAP", "MSSQL", "NetBIOS"],
            ["SSDP", "TFTP"],
            ["Portmap"]]

# ====== SỬA NHÃN (GĐ3) ======
# UDP-lag "thật" (UDP, trong cửa sổ 13:11-13:15) chỉ ~2% số dòng UDPLag ngày 1 và không tách được khỏi UDP (AUC 0.51)
MERGE_LABELS = {"UDPLag": "UDP"}

# CIC-DDoS2019 gán nhãn theo CỬA SỔ THỜI GIAN -> flow TCP trong cửa sổ UDP-lag bị gán UDPLag (thực chất là SYN).
FIX_PROTOCOL_LABELS = True
UDP_BASED = {"UDP", "UDPLag", "DNS", "NTP", "SNMP", "LDAP", "MSSQL", "NetBIOS", "SSDP", "TFTP", "Portmap"}
TCP_BASED = {"SYN"}

# 3b. vector giống hệt nhưng khác nhãn
CONFLICT_POLICY     = "majority"   # "drop": loại mọi vector khác nhãn (cách cũ, Portmap còn ~500 dòng)
                                   # "majority": giữ dòng của nhãn đa số nếu nhãn đó chiếm >= CONFLICT_MIN_PURITY
CONFLICT_MIN_PURITY = 0.9
CONFLICT_SCOPE      = "all"        # "all": xét trùng trên cả hai ngày | "day": chỉ xét trong cùng một ngày

META_COLS = ["label", "day", "ts", "ord", "ts_kind", "_h"]   # _h: băm dòng để lọc trùng giữa file (bỏ sau GĐ3)
# ====== LƯU/ĐỌC: dùng parquet nếu có pyarrow (Kaggle có sẵn), không thì pickle ======
try:
    import pyarrow  # noqa
    EXT = ".parquet"
    def save_df(df, path): df.to_parquet(path.with_suffix(EXT), index=False)
    def load_df(path):     return pd.read_parquet(path)
except ImportError:
    EXT = ".pkl"
    def save_df(df, path): df.to_pickle(path.with_suffix(EXT))
    def load_df(path):     return pd.read_pickle(path)

def canon_label(x):
    k = re.sub(r"[^a-z0-9]", "", str(x).lower().replace("drdos_", ""))
    return LABEL_CANON.get(k)

def map_labels(series):
    """Ánh xạ nhanh trên giá trị duy nhất (không gọi hàm trên hàng triệu dòng)."""
    mp = {u: canon_label(u) for u in series.astype(str).unique()}
    return series.astype(str).map(mp)

print("INPUT_DIR:", INPUT_DIR, "| WORK_DIR:", WORK_DIR, "| định dạng trung gian:", EXT)

## GIAI ĐOẠN 1 - Tìm file CSV và gán ngày

In [ ]:
# %%
if not RUN_PREPROCESS:
    print('Bỏ qua (RUN_PREPROCESS=False)')
else:
    def day_of(path):
        p = str(path).lower()
        hits = [d for d, pats in DAY_PATTERNS.items() if any(re.search(x, p) for x in pats)]
        return hits[0] if len(hits) == 1 else None

    all_csv = sorted(INPUT_DIR.rglob("*.csv"))
    files_ok, files_bad = [], []
    for f in all_csv:
        (files_ok if day_of(f) else files_bad).append(f)

    print(f"Tìm thấy {len(all_csv)} file CSV; gán được ngày cho {len(files_ok)} file.")
    for f in files_ok:
        print(f"  [{day_of(f)}] {f}")
    if files_bad:
        print("\nCHƯA xác định được ngày (sửa DAY_PATTERNS ở Giai đoạn 0 nếu cần):")
        for f in files_bad:
            print("  ", f)
    assert files_ok or (REUSE_INTERIM and any(INTERIM.glob(f"*{EXT}"))), \
        "Không có file nào! Kiểm tra cell COLAB 4/5 (dataset đã giải nén vào INPUT_DIR chưa)."
    if not files_ok:
        print("Không cần CSV: dùng lại dữ liệu trung gian trên Drive (REUSE_INTERIM=True).")

## GIAI ĐOẠN 2 - Đọc CSV theo chunk, chuẩn hóa nhãn, loại trùng, lấy mẫu đều, lưu trung gian

In [ ]:
# ===== GIAI ĐOẠN 2 - Đọc CSV theo chunk, chuẩn hóa nhãn, LOẠI TRÙNG, rồi lấy mẫu đều, lưu trung gian =====
if not RUN_PREPROCESS:
    print('Bỏ qua (RUN_PREPROCESS=False)')
else:
    if REUSE_INTERIM and any(INTERIM.glob(f'*{EXT}')):
        print(f'Dùng lại dữ liệu trung gian trong {INTERIM} (REUSE_INTERIM=True) - bỏ qua bước đọc CSV.')
    else:
        from pandas.util import hash_pandas_object

        def header_info(path):
            cols = list(pd.read_csv(path, nrows=0).columns)
            label_col = next(c for c in cols if c.strip().lower() == "label")
            ts_col = next((c for c in cols if c.strip().lower() == "timestamp"), None)
            return cols, label_col, ts_col

        # ---- Pass 1: đếm số dòng mỗi nhãn trong từng file (chỉ đọc cột Label nên nhanh) ----
        t0 = time.time()
        file_counts = {}
        for f in files_ok:
            _, lc, _ = header_info(f)
            cnt = {}
            for i, ch in enumerate(pd.read_csv(f, usecols=[lc], chunksize=2_000_000)):
                vc = map_labels(ch[lc]).fillna("_dropped_").value_counts()
                for k, v in vc.items():
                    cnt[k] = cnt.get(k, 0) + int(v)
                if DEBUG_MAX_CHUNKS and i + 1 >= DEBUG_MAX_CHUNKS:
                    break
            file_counts[str(f)] = cnt
            print(f"{f.name:<40} {cnt}")
        print(f"Pass 1 xong trong {time.time()-t0:.0f}s")
        # ---- Pass 2: đọc đầy đủ, lấy mẫu đều (giữ phân bố thời gian), làm sạch, ghi trung gian ----
        def clean_chunk(ch, keep, lab, tscol, lc):
            sub = ch.loc[keep]
            out = pd.DataFrame({
                "label": lab[keep].values,
                "ord": sub.index.values
            }, index=sub.index)

            if tscol is not None:
                t = pd.to_datetime(sub[tscol], errors="coerce")
                out["ts_time"] = (
                    t - pd.Timestamp("1970-01-01")
                ).dt.total_seconds().values
            else:
                out["ts_time"] = np.nan

            # băm theo MỌI trường của dòng CSV gốc (trừ cột chỉ số), tính TRƯỚC khi bỏ cột định danh
            h_all = None
            if DEDUP_BEFORE_SAMPLE and DEDUP_MODE == "all_fields":
                hcols = [c for c in sub.columns if c.strip().lower() != "unnamed: 0"]
                h_all = hash_pandas_object(sub[hcols], index=False).values

            feats = sub.drop(
                columns=[
                    c for c in sub.columns
                    if c.strip().lower() in DROP_SET
                    or c == lc
                    or c == tscol
                ]
            )

            feats.columns = [c.strip() for c in feats.columns]
            feats = feats.apply(pd.to_numeric, errors="coerce")
            if INF_NAN_MODE == "drop":
                feats = feats.replace([np.inf, -np.inf], np.nan)

            out = pd.concat([out, feats], axis=1)
            if h_all is not None:
                out["_h"] = h_all

            if INF_NAN_MODE == "drop":
                out = out.dropna(subset=[c for c in out.columns if c not in ("ts_time",)])
            else:
                out = out.dropna(subset=["label"])          # GIỮ inf/NaN trong đặc trưng (xử lý ở GĐ3 + GĐ8)

            fc = list(feats.columns)

            if DEDUP_BEFORE_SAMPLE and h_all is None:      # DEDUP_MODE == "features": băm vector đặc trưng (cách cũ)
                out["_h"] = hash_pandas_object(
                    out[fc],
                    index=False
                ).values

            out[fc] = out[fc].astype("float32")

            return out


        def dedup_chunk(c, day, seen):
            """Loại dòng trùng vector đặc trưng trong (ngày, nhãn),
            xuyên suốt các chunk, bằng băm 64-bit.
            """
            c = c.drop_duplicates(subset=["label", "_h"])

            keep = np.ones(len(c), dtype=bool)

            labs = c["label"].to_numpy()
            hs = c["_h"].to_numpy()

            for lab in np.unique(labs):
                idx = np.where(labs == lab)[0]

                s = seen.setdefault((day, lab), set())

                hl = hs[idx].tolist()

                is_new = np.fromiter(
                    (x not in s for x in hl),
                    dtype=bool,
                    count=len(hl)
                )

                s.update(
                    np.array(hl, dtype=np.uint64)[is_new].tolist()
                )

                keep[idx[~is_new]] = False

            return c.loc[keep]                     # giữ _h để GĐ3 lọc trùng GIỮA các file


        t0 = time.time()

        for old in INTERIM.glob("*"):
            old.unlink()

        written, seen = [], {}

        for f in files_ok:
            day = day_of(f)
            _, lc, tscol = header_info(f)
            cnt = file_counts[str(f)]

            p_lab = {
                lab: (
                    1.0
                    if lab == BENIGN
                    else min(1.0, PRE_CAP_ATTACK / n)
                )
                for lab, n in cnt.items()
                if lab != "_dropped_"
            }

            parts, n_in, n_raw, n_uni = [], 0, 0, 0

            for i, ch in enumerate(
                pd.read_csv(
                    f,
                    chunksize=CHUNK,
                    low_memory=False
                )
            ):
                lab = map_labels(ch[lc])

                if DEDUP_BEFORE_SAMPLE:
                    # Giữ hết, loại trùng trước
                    keep = lab.notna().to_numpy()

                else:
                    p = lab.map(p_lab).astype(float)

                    keep = (
                        (rng.random(len(ch)) < p.fillna(0).values)
                        & lab.notna().to_numpy()
                    )

                if keep.any():
                    n_in += int(keep.sum())

                    c = clean_chunk(
                        ch,
                        keep,
                        lab,
                        tscol,
                        lc
                    )

                    n_raw += len(c)

                    if DEDUP_BEFORE_SAMPLE:
                        c = dedup_chunk(
                            c,
                            day,
                            seen
                        )

                    n_uni += len(c)

                    if len(c):
                        parts.append(c)

                if DEBUG_MAX_CHUNKS and i + 1 >= DEBUG_MAX_CHUNKS:
                    break

            if not parts:
                continue

            df = pd.concat(
                parts,
                ignore_index=True
            )

            if DEDUP_BEFORE_SAMPLE:
                # Lấy mẫu đều trong số các dòng duy nhất
                # -> vẫn phủ hết cửa sổ thời gian
                sampled = []

                for lab, g in df.groupby("label"):
                    if lab != BENIGN and len(g) > PRE_CAP_ATTACK:
                        g = g.sample(
                            PRE_CAP_ATTACK,
                            random_state=SEED
                        )

                    sampled.append(g)

                df = pd.concat(
                    sampled,
                    ignore_index=True
                )

                # Giải phóng bộ nhớ
                # Benign có mặt ở mọi file nên không xóa theo cách này
                for k in [
                    k for k in seen
                    if k[1] != BENIGN
                ]:
                    del seen[k]

            # Thời gian:
            # dùng Timestamp nếu parse được >= 99%,
            # nếu không dùng thứ tự dòng trong file
            ok = df["ts_time"].notna().mean() >= 0.99

            df["ts_kind"] = "time" if ok else "order"

            df["ts"] = (
                df["ts_time"]
                if ok
                else df["ord"].astype("float64")
            )

            df = (
                df
                .dropna(subset=["ts"])
                .drop(columns=["ts_time"])
            )

            df["day"] = day

            if not ok:
                print(
                    f"  CẢNH BÁO: {f.name}: "
                    f"Timestamp không parse được, "
                    f"dùng thứ tự dòng."
                )

            for lab, g in df.groupby("label"):
                out = INTERIM / f"{day}__{lab}__{f.stem}"

                save_df(
                    g.reset_index(drop=True),
                    out
                )

                written.append(
                    (
                        day,
                        lab,
                        f.name,
                        len(g)
                    )
                )

            extra = (
                f" | đọc {n_in:,}"
                f" -> bỏ {'NaN/inf' if INF_NAN_MODE == 'drop' else 'thiếu nhãn'} {n_in-n_raw:,}"
                f" -> còn {n_raw:,}"
                f" -> duy nhất {n_uni:,}"
                f" ({100*(1-n_uni/max(n_raw,1)):.1f}% trùng)"
            ) if DEDUP_BEFORE_SAMPLE else ""

            print(
                f"{f.name:<40} "
                f"giữ lại {len(df):>9,} dòng"
                f"{extra}"
            )

            del parts, df
            gc.collect()


        print(
            f"Pass 2 xong trong "
            f"{time.time()-t0:.0f}s; "
            f"{len(written)} file trung gian."
        )

## GIAI ĐOẠN 3 - Gộp, sửa nhãn, loại trùng, xử lý vector mâu thuẫn nhãn, loại cột hằng/trùng

In [ ]:
# ===== GIAI ĐOẠN 3 - Gộp, loại trùng, loại dòng mâu thuẫn nhãn, loại cột hằng/trùng =====
if not RUN_PREPROCESS:
    print('Bỏ qua (RUN_PREPROCESS=False)')
else:
    from pandas.util import hash_pandas_object

    full = pd.concat([load_df(p) for p in sorted(INTERIM.glob(f"*{EXT}"))], ignore_index=True)
    feats = [c for c in full.columns if c not in META_COLS]
    print(f"Sau khi gộp: {len(full):,} dòng, {len(feats)} đặc trưng")
    if MERGE_LABELS:
        _n = int(full["label"].isin(list(MERGE_LABELS)).sum())
        full["label"] = full["label"].astype(str).replace(MERGE_LABELS)
        print(f"MERGE_LABELS {MERGE_LABELS}: đổi nhãn {_n:,} dòng")
    if DEDUP_VECTORS and "_h" not in full.columns:
        raise RuntimeError("File trung gian cũ (chưa có cột _h): đặt REUSE_INTERIM=False và chạy lại từ GIAI ĐOẠN 2")

    # Lọc nhãn theo cửa sổ tấn công chính thức (nếu khai báo ATTACK_WINDOWS ở GĐ0)
    for (d_, lab_), (a_, b_) in ATTACK_WINDOWS.items():
        m_ = ((full["day"] == d_) & (full["label"] == lab_)).to_numpy()
        t_ = pd.to_datetime(full.loc[m_, "ts"], unit="s")
        bad_ = m_.copy(); bad_[m_] = ~((t_ >= a_) & (t_ < b_)).to_numpy()
        print(f"Lọc cửa sổ {lab_}/{d_}: giữ {int(m_.sum() - bad_.sum()):,} / {int(m_.sum()):,} dòng")
        full = full.loc[~bad_].reset_index(drop=True)

    # 3a. trùng còn sót GIỮA các file (cùng một dòng xuất hiện ở nhiều CSV, vd. Benign); trong file đã xử lý ở GĐ2
    n0 = len(full)
    if DEDUP_VECTORS:
        full = full.sort_values(["day", "ts"]).drop_duplicates(subset=["label", "_h"], keep="first")
        full = full.reset_index(drop=True)
    if "_h" in full.columns:
        full = full.drop(columns="_h")
    print(f"3a. Loại trùng giữa file (DEDUP_MODE={DEDUP_MODE}): {n0:,} -> {len(full):,}  (bỏ {n0-len(full):,})")

    # 3a+. lọc nhãn theo giao thức: tấn công dựa trên UDP phải là UDP (17), SYN phải là TCP (6)
    if FIX_PROTOCOL_LABELS:
        _proto = full["Protocol"].to_numpy()
        _lab = full["label"].to_numpy()
        _bad = ((np.isin(_lab, list(UDP_BASED)) & (_proto != 17)) |
                (np.isin(_lab, list(TCP_BASED)) & (_proto != 6)))
        _rep = (full.loc[_bad].assign(Protocol=full.loc[_bad, "Protocol"].astype(int))
                    .groupby(["day", "label", "Protocol"]).size().rename("so_dong_bi_loai"))
        print(f"3a+. Loại {int(_bad.sum()):,} dòng có giao thức trái với loại tấn công:")
        print(_rep.to_string() if len(_rep) else "   (không có)")
        full = full.loc[~_bad].reset_index(drop=True)

    # 3b. dòng có đặc trưng giống hệt nhưng KHÁC nhãn
    #     CONFLICT_POLICY="drop"    : loại hết (cách cũ)
    #     CONFLICT_POLICY="majority": giữ dòng của nhãn chiếm >= CONFLICT_MIN_PURITY trong vector đó, loại phần còn lại
    _key_cols = feats + (["day"] if CONFLICT_SCOPE == "day" else [])
    _h = hash_pandas_object(full[_key_cols], index=False).to_numpy()
    _rows = pd.DataFrame({"h": _h, "label": full["label"].to_numpy()})
    _cnt = _rows.groupby(["h", "label"], sort=False).size().rename("n").reset_index()
    _cnt["tot"] = _cnt.groupby("h")["n"].transform("sum")
    _cnt["n_lab"] = _cnt.groupby("h")["label"].transform("size")
    _top = (_cnt.sort_values("n", ascending=False).drop_duplicates("h")
                .assign(top_purity=lambda t: t["n"] / t["tot"])[["h", "label", "top_purity"]]
                .rename(columns={"label": "top"}))
    _info = _rows.merge(_cnt[["h", "label", "n_lab"]], on=["h", "label"], how="left").merge(_top, on="h", how="left")
    conflict = (_info["n_lab"] > 1).to_numpy()
    keep_major = ((_info["label"] == _info["top"]).to_numpy() &
                  (_info["top_purity"] >= CONFLICT_MIN_PURITY).to_numpy())
    print(f"3b. Dòng có vector giống hệt nhưng khác nhãn: {int(conflict.sum()):,}  "
          f"(CONFLICT_POLICY={CONFLICT_POLICY}, MIN_PURITY={CONFLICT_MIN_PURITY}, SCOPE={CONFLICT_SCOPE})")
    _lost = _info.loc[conflict & ~keep_major]
    _diag = (_lost.assign(dung_voi=np.where(_lost["label"] == _lost["top"], "(đa số nhưng chưa đủ thuần)", _lost["top"]))
                  .groupby(["label", "dung_voi"]).size().rename("so_dong").reset_index()
                  .sort_values(["label", "so_dong"], ascending=[True, False]).groupby("label").head(3))
    print("   Dòng bị loại theo lớp (3 lớp đụng nhiều nhất):\n" + _diag.to_string(index=False))
    if not DROP_CONFLICTS:
        _drop = np.zeros(len(full), bool)
    elif CONFLICT_POLICY == "drop":
        _drop = conflict
    elif CONFLICT_POLICY == "majority":
        _drop = conflict & ~keep_major
    else:
        raise ValueError(f"CONFLICT_POLICY không hợp lệ: {CONFLICT_POLICY}")
    print(f"   Giữ lại {int((conflict & ~_drop).sum()):,} dòng xung đột, loại {int(_drop.sum()):,}")
    full = full.loc[~_drop].reset_index(drop=True)
    del _rows, _cnt, _top, _info, _lost

    # 3c. cột hằng số
    nun = full[feats].nunique(dropna=(INF_NAN_MODE == "drop"))   # inf/NaN cũng tính là một giá trị
    const_cols = nun[nun <= 1].index.tolist()
    print(f"3c. Cột hằng số ({len(const_cols)}): {const_cols}")
    feats = [c for c in feats if c not in const_cols]

    # 3d. cột trùng nội dung (vd. 'Fwd Header Length' và 'Fwd Header Length.1')
    sig, dup_cols = {}, []
    for c in feats:
        s = int(hash_pandas_object(full[c], index=False).values.sum(dtype=np.uint64))
        if s in sig:
            dup_cols.append((c, sig[s]))
        else:
            sig[s] = c
    print(f"3d. Cột trùng nội dung: {dup_cols}")
    feats = [c for c in feats if c not in {a for a, _ in dup_cols}]

    full = full[feats + [c for c in META_COLS if c in full.columns]]
    print(f"Còn lại {len(full):,} dòng x {len(feats)} đặc trưng")
    print(full.groupby(["day", "label"]).size().unstack(0).fillna(0).astype(int))

    # 3e. cờ inf / missing: chỉ tạo cho cột THỰC SỰ có giá trị đó (cờ được dựng lại từ giá trị thô ở GĐ8 và trên server)
    INF_FLAGS = []                                  # [(tên cờ, cột gốc, loại)]
    if INF_NAN_MODE == "flag_impute":
        rows_ = []
        for c in feats:
            v = full[c].to_numpy()
            for kind, m in (("posinf", np.isposinf(v)), ("neginf", np.isneginf(v)), ("missing", np.isnan(v))):
                k = int(m.sum())
                if k:
                    INF_FLAGS.append((f"{c}__is_{kind}", c, kind))
                    rows_.append(dict(cot=c, loai=kind, so_dong=k, ti_le=round(k / len(full), 5)))
        n_any = int((~np.isfinite(full[feats].to_numpy(dtype="float64"))).any(axis=1).sum())
        print(f"\n3e. Dòng có ít nhất một inf/NaN: {n_any:,} ({100 * n_any / len(full):.2f}%) -> GIỮ LẠI, tạo {len(INF_FLAGS)} cờ:")
        if rows_:
            print(pd.DataFrame(rows_).to_string(index=False))

## GIAI ĐOẠN 4 - Giới hạn số mẫu mỗi (ngày, lớp)

In [ ]:
# ===== GIAI ĐOẠN 4 - Giới hạn số mẫu mỗi (ngày, lớp); giữ toàn bộ Benign =====
if not RUN_PREPROCESS:
    print('Bỏ qua (RUN_PREPROCESS=False)')
else:
    if "full" not in globals():
        raise RuntimeError("Chưa có biến `full`: hãy chạy lại GIAI ĐOẠN 3 rồi mới chạy GIAI ĐOẠN 4.")
    if CAP_PER_DAY_LABEL is None:
        df = full                                                       # dùng MỌI dòng sau lọc
    else:
        capped = []
        for (day, lab), g in full.groupby(["day", "label"], sort=False):
            if lab != BENIGN and len(g) > CAP_PER_DAY_LABEL:
                g = g.sample(CAP_PER_DAY_LABEL, random_state=SEED)   # lấy mẫu đều -> vẫn phủ hết cửa sổ thời gian
            capped.append(g)
        df = pd.concat(capped, ignore_index=True)
        del capped
    del full; gc.collect()
    print(f"Sau giới hạn (CAP_PER_DAY_LABEL={CAP_PER_DAY_LABEL}): {len(df):,} dòng")
    print(df.groupby(["day", "label"]).size().unstack(0).fillna(0).astype(int))

## GIAI ĐOẠN 5 - Chia train/val/test theo THỨ TỰ THỜI GIAN trong từng (ngày, lớp)

In [ ]:
# %%
if not RUN_PREPROCESS:
    print('Bỏ qua (RUN_PREPROCESS=False)')
else:
    def _interleave_pattern(frac, n_blocks):
        """Gán xen kẽ các khối cho train/val/test sao cho tỉ lệ ~ frac, val/test nằm rải đều theo thời gian."""
        n_te = max(1, round(frac[2] * n_blocks)); n_va = max(1, round(frac[1] * n_blocks))
        pat = ["train"] * n_blocks
        for k in range(n_te):
            pat[int((k + 1) * n_blocks / n_te) - 1] = "test"
        free = [i for i in range(n_blocks) if pat[i] == "train"]
        for k in range(n_va):
            pat[free[int((k + 0.5) * len(free) / n_va)]] = "val"
        return pat

    def block_split(df, frac=SPLIT_FRAC, gap_s=GAP_S, mode=SPLIT_MODE, n_blocks=N_BLOCKS, max_gap_frac=MAX_GAP_FRAC):
        """
        Mỗi (ngày, lớp): sắp theo thời gian rồi chia theo SỐ DÒNG (phân vị), KHÔNG theo tỉ lệ khoảng thời gian.
        (Chia theo khoảng thời gian sai khi vài dòng có timestamp ngoại lai kéo giãn khoảng: toàn bộ dữ liệu có thể
         rơi vào một khối, ví dụ NTP train=59/val=5/test=29,936.)
        Ở mỗi ranh giới giữa hai khối khác loại, cắt các dòng nằm trong gap_s giây quanh ranh giới ở CẢ HAI phía
        (tối đa max_gap_frac * số dòng mỗi phía) để flow dài không 'tràn' qua ranh giới.
        """
        split = np.full(len(df), "drop", dtype=object)
        for (day, lab), g in df.groupby(["day", "label"]):
            order = np.argsort(g["ts"].to_numpy(), kind="stable")
            idx = g.index.to_numpy()[order]
            t = g["ts"].to_numpy()[order]
            n = len(idx)
            gap = gap_s if (g["ts_kind"] == "time").all() else 0.0     # ts là thứ tự dòng -> chỉ dùng trần theo số dòng
            if mode == "contiguous":
                cuts = [0, int(round(frac[0] * n)), int(round((frac[0] + frac[1]) * n)), n]
                blocks = [(cuts[i], cuts[i + 1], k) for i, k in enumerate(["train", "val", "test"])]
            elif mode == "interleaved":
                edges = np.linspace(0, n, n_blocks + 1).round().astype(int)
                pat = _interleave_pattern(frac, n_blocks)
                blocks = [(edges[i], edges[i + 1], pat[i]) for i in range(n_blocks)]
            else:
                raise ValueError(f"SPLIT_MODE={mode!r}")
            s = np.empty(n, dtype=object)
            for a, b, k in blocks:
                s[a:b] = k
            # Ngân sách cắt tổng cộng ~ 4 * max_gap_frac * n (đúng bằng chế độ contiguous với 2 ranh giới x 2 phía),
            # chia đều cho các ranh giới khác loại: chế độ xen kẽ có nhiều ranh giới hơn nên mỗi phía bị cắt ít hơn.
            n_bound = max(1, sum(1 for x, y2 in zip(blocks[:-1], blocks[1:]) if x[2] != y2[2]))
            cap = max(1, int(np.ceil(4 * max_gap_frac * n / (2 * n_bound))))
            for (a1, b1, k1), (a2, b2, k2) in zip(blocks[:-1], blocks[1:]):
                if k1 == k2 or b1 >= n or b1 <= a1:
                    continue
                tb = t[b1]                                               # thời điểm của dòng đầu khối sau
                j, c = b1 - 1, 0                                         # đuôi khối trước
                while j >= a1 and c < cap and t[j] > tb - gap:
                    s[j] = "drop"; j -= 1; c += 1
                j, c = b1, 0                                             # đầu khối sau
                while j < b2 and c < cap and t[j] < tb + gap:
                    s[j] = "drop"; j += 1; c += 1
            split[df.index.get_indexer(idx)] = s
            cnt = {k: int((s == k).sum()) for k in ("train", "val", "test")}
            if cnt["test"] < 30 or cnt["train"] < 100:
                print(f"  CẢNH BÁO: {day}/{lab} quá ít mẫu sau khi chia: {cnt}")
        return split

    df["split"] = block_split(df)
    print(f"SPLIT_MODE={SPLIT_MODE} | dòng bị bỏ do khoảng đệm:", int((df["split"] == "drop").sum()))
    df = df[df["split"] != "drop"].reset_index(drop=True)
    _cnt = df.groupby(["label", "day", "split"]).size().unstack([1, 2]).fillna(0).astype(int)
    print(_cnt)
    # Kiểm tra chất lượng: tỉ lệ train/val/test của MỖI (ngày, lớp) phải xấp xỉ 0.6/0.2/0.2
    _share = df.groupby(["day", "label"])["split"].value_counts(normalize=True).unstack(fill_value=0)
    _bad = _share[(_share.get("train", 0) < 0.4) | (_share.get("val", 0) < 0.08) | (_share.get("test", 0) < 0.08)]
    if len(_bad) and SPLIT_MODE == "contiguous":
        print("CẢNH BÁO: tỉ lệ chia lệch mạnh so với 0.6/0.2/0.2:\n", _bad.round(3))
    else:
        print("Kiểm tra tỉ lệ chia: đạt (mỗi lớp xấp xỉ 0.6/0.2/0.2)")

## GIAI ĐOẠN 6 - Kiểm tra rò rỉ gần-trùng giữa các tập

In [ ]:
# ===== GIAI ĐOẠN 6 - Kiểm tra rò rỉ gần-trùng giữa các tập (tùy chọn: xóa) =====
if not RUN_PREPROCESS:
    print('Bỏ qua (RUN_PREPROCESS=False)')
else:
    def signed_log(a):
        return np.sign(a) * np.log1p(np.abs(a))

    def near_hash(frame):
        arr = np.round(signed_log(frame[feats].to_numpy(dtype="float64")), NEAR_DEC)
        return pd.util.hash_pandas_object(pd.DataFrame(arr), index=False).to_numpy()

    hh = near_hash(df)
    is_train = (df["split"] == "train").to_numpy()
    is_dup = np.isin(hh, hh[is_train]) & ~is_train          # val/test gần-trùng một dòng train
    rep = (df.loc[~is_train, ["label", "split"]].assign(near_dup=is_dup[~is_train])
             .groupby(["label", "split"])["near_dup"].mean().unstack().round(3))
    print("Tỉ lệ dòng val/test gần-trùng một dòng train (làm tròn %d chữ số sau signed-log):" % NEAR_DEC)
    print(rep)
    print("Tỉ lệ cao (>0.5) nghĩa là lớp đó rất 'lặp' -> điểm test sẽ lạc quan; hãy ghi chú trong báo cáo.")
    if PURGE_NEAR_DUP:
        df = df[~is_dup].reset_index(drop=True)
        print("Đã xóa gần-trùng; còn", len(df), "dòng")

    # Rò rỉ do BẢN SAO: dòng val/test TRÙNG HỆT một dòng train CÙNG lớp (quan trọng khi FILL_TO_CAP=True)
    hx = pd.util.hash_pandas_object(df[feats], index=False).to_numpy()
    kx = pd.util.hash_pandas_object(pd.DataFrame({"l": df["label"].to_numpy(), "h": hx}), index=False).to_numpy()
    is_tr = (df["split"] == "train").to_numpy()
    exact = np.isin(kx, kx[is_tr]) & ~is_tr
    rep_x = (df.loc[~is_tr, ["label", "split"]].assign(exact_dup=exact[~is_tr])
               .groupby(["label", "split"])["exact_dup"].mean().unstack().round(3))
    print("\nTỉ lệ dòng val/test TRÙNG HỆT một dòng train cùng lớp (ghi vào báo cáo nếu cao):")
    print(rep_x)
    if PURGE_EXACT_DUP and exact.any():
        df = df[~exact].reset_index(drop=True)
        print(f"PURGE_EXACT_DUP: đã xóa {int(exact.sum()):,} dòng val/test trùng train; còn {len(df):,} dòng")
        print(df.groupby(["label", "split"]).size().unstack().fillna(0).astype(int))

## GIAI ĐOẠN 7 - Mã hóa lớp, định nghĩa stage cho kịch bản A và B

In [ ]:
# %%
if not RUN_PREPROCESS:
    print('Bỏ qua (RUN_PREPROCESS=False)')
else:
    present = set(df["label"].unique())
    order = [BENIGN] + [c for st in STAGES_A for c in st]
    classes = [c for c in order if c in present] + sorted(present - set(order))
    label2id = {c: i for i, c in enumerate(classes)}
    df["y"] = df["label"].map(label2id).astype(np.int16)
    print("Lớp (theo id):", label2id)

    stage_of = {c: i for i, st in enumerate(STAGES_A) for c in st}
    for i, st in enumerate(STAGES_A):
        miss = [c for c in st if c not in present]
        print(f"Stage A{i}: {st}" + (f"   THIẾU trong dữ liệu: {miss}" if miss else ""))

    day1_cls = set(df.loc[df["day"] == "day1", "label"])
    day2_cls = set(df.loc[df["day"] == "day2", "label"])
    is_att_in_A = df["label"].isin(stage_of.keys())
    is_benign = df["label"] == BENIGN
    n_st = len(STAGES_A)

    # ---- Kịch bản A: lớp tấn công lấy từ ngày 1; lớp chỉ có ở ngày 2 (vd. Portmap) lấy từ ngày 2 ----
    A_use = (
        (is_benign & (df["day"] == "day1")) |
        (is_att_in_A & df["label"].isin(day1_cls) & (df["day"] == "day1")) |
        (is_att_in_A & ~df["label"].isin(day1_cls) & (df["day"] == "day2"))
    ).to_numpy()
    A_stage = np.full(len(df), -1, dtype=np.int8)
    m = is_att_in_A.to_numpy()
    A_stage[m] = df.loc[m, "label"].map(stage_of).to_numpy()
    # Benign: chia train/val thành n_st phần rời nhau (mỗi stage có benign mới); benign test = -1 (dùng ở mọi stage)
    mb = (is_benign & df["split"].isin(["train", "val"]) & (df["day"] == "day1")).to_numpy()
    A_stage[mb] = rng.integers(0, n_st, size=mb.sum())
    A_stage[~A_use] = -1

    # ---- Kịch bản B: các lớp có mặt ở CẢ HAI ngày; stage 0 = ngày 1, stage 1 = ngày 2 ----
    cnt = df.groupby(["day", "label"]).size().unstack(0).fillna(0)
    for d in ("day1", "day2"):
        if d not in cnt.columns:
            cnt[d] = 0
    shared = set(cnt.index[(cnt["day1"] >= MIN_SHARED_PER_DAY) & (cnt["day2"] >= MIN_SHARED_PER_DAY)])
    too_small = sorted((day1_cls & day2_cls) - shared)
    if too_small:
        print(f"CẢNH BÁO: lớp có ở cả hai ngày nhưng < {MIN_SHARED_PER_DAY} mẫu/ngày, bị loại khỏi kịch bản B: "
              f"{ {c: (int(cnt.loc[c,'day1']), int(cnt.loc[c,'day2'])) for c in too_small} }")
    B_stage = np.full(len(df), -1, dtype=np.int8)
    for k, d in enumerate(["day1", "day2"]):
        B_stage[((df["day"] == d) & df["label"].isin(shared)).to_numpy()] = k
    print("Lớp dùng chung hai ngày (kịch bản B):", sorted(shared))
    print("Lớp chỉ có ngày 1:", sorted(day1_cls - day2_cls), "| chỉ có ngày 2:", sorted(day2_cls - day1_cls))

## TOKENIZER (tabular_tokenizer.py - Phase 1-3) + TOKENIZER v2

In [ ]:
# %%
# ======================================================================================
# TOKENIZER: sao chép từ tabular_tokenizer.py của bạn (Phase 1 profiling, Phase 2 grouping,
# Phase 3 scale+tokenize). Giữ nguyên thuật toán; bỏ phần đọc CSV/chia ngẫu nhiên vì dữ liệu
# và split đã do các giai đoạn trước tạo ra.
# ======================================================================================
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import QuantileTransformer, LabelEncoder
from sklearn.model_selection import train_test_split


warnings.filterwarnings("ignore")

_VAL_TEST_FRACTION         = 0.30
_TEST_WITHIN_TEMP_FRACTION = 0.50
_SPLIT_RANDOM_STATE        = 42
DEFAULT_SCALE_FACTOR       = 30_000
_PORT_VOCAB                = 7
_EPS                       = 1e-9
_PERCENTILES               = [0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 100]

_CLIP_LO_PCT = 0.1
_CLIP_HI_PCT = 99.9

GROUP_BINARY  = "binary"
GROUP_PORT    = "port"
GROUP_LOWCARD = "low_card"
GROUP_SKEWED  = "skewed"
GROUP_NORMAL  = "normal"

_ALL_GROUPS = [GROUP_BINARY, GROUP_PORT, GROUP_LOWCARD, GROUP_SKEWED, GROUP_NORMAL]

STRAT_LINEAR    = "robust_linear"
STRAT_QUANTILE  = "quantile_rank"
STRAT_SQRT      = "signed_sqrt"
STRAT_CBRT      = "signed_cbrt"
STRAT_PIECEWISE = "piecewise_sqrt"

FORCE_QUANTILE_FEATURES = {
    "packets_IAT_mode",
    "packets_IAT_median",
    "fwd_packets_IAT_median",
    "fwd_packets_IAT_mean",
    "packet_IAT_total",
    "packet_IAT_max",
}


def _profile_one(col_name: str, series: pd.Series) -> dict:
    v = series.dropna().values.astype(float)
    n = len(v)

    pct = {f"p{p}": (float(np.percentile(v, p)) if n > 0 else 0.0)
           for p in _PERCENTILES}

    p1, p25, p50, p75, p90, p99 = (
        pct["p1"], pct["p25"], pct["p50"],
        pct["p75"], pct["p90"], pct["p99"]
    )
    iqr           = p75 - p25
    body          = p99 - p1
    tail_ratio    = (p99 - p50) / (p50 - p1  + _EPS)
    tail_75_ratio = (p99 - p75) / (iqr        + _EPS)
    tail_90_ratio = (p99 - p90) / (p90 - p50  + _EPS)
    left_ratio    = (p50 - p1)  / (body        + _EPS)
    spread_ratio  = iqr          / (body        + _EPS)
    zero_ratio    = float((v == 0).mean()) if n > 0 else 0.0
    n_unique      = int(np.unique(v).size)

    clip_lo = float(np.percentile(v, _CLIP_LO_PCT)) if n > 0 else 0.0
    clip_hi = float(np.percentile(v, _CLIP_HI_PCT)) if n > 0 else 0.0

    return {
        "feature":       col_name,
        "clip_lo":       round(clip_lo, 6),
        "clip_hi":       round(clip_hi, 6),
        **{k: round(val, 6) for k, val in pct.items()},
        "iqr":           round(iqr,           6),
        "tail_ratio":    round(tail_ratio,    4),
        "tail_75_ratio": round(tail_75_ratio, 4),
        "tail_90_ratio": round(tail_90_ratio, 4),
        "left_ratio":    round(left_ratio,    4),
        "spread_ratio":  round(spread_ratio,  4),
        "zero_ratio":    round(zero_ratio,    4),
        "n_unique":      n_unique,
        "mean":          round(float(np.mean(v)),  6) if n > 0 else 0.0,
        "std":           round(float(np.std(v)),   6) if n > 0 else 0.0,
        "skew":          round(float(pd.Series(v).skew()),     4) if n > 2 else 0.0,
        "kurtosis":      round(float(pd.Series(v).kurtosis()), 4) if n > 3 else 0.0,
        "cv":            round(float(np.std(v) / (np.mean(v) + _EPS)), 4) if n > 0 else 0.0,
        "is_binary":     set(np.unique(v)).issubset({0.0, 1.0}),
        "is_low_card":   n_unique < 20,
        "is_port":       "port" in col_name.lower(),
    }


def phase1_profile(df: pd.DataFrame,
                   features: list,
                   train_idx: np.ndarray) -> dict:
    print(f"\n[Phase 1] Quantile-profiling {len(features)} features "
          f"({len(train_idx):,} train rows) …")
    profiles = {col: _profile_one(col, df.loc[train_idx, col]) for col in features}
    print(f"          Percentiles : {_PERCENTILES}")
    print(f"          Clip bounds : p{_CLIP_LO_PCT} – p{_CLIP_HI_PCT}  (train-fitted)")
    return profiles


_GROUP_CRITERIA = {
    GROUP_BINARY:  "unique ⊆ {0,1}",
    GROUP_PORT:    "col name contains 'port'",
    GROUP_LOWCARD: "n_unique < 20",
    GROUP_SKEWED:  "tail_ratio > 5  OR  spread_ratio < 0.05  OR  left_ratio extreme",
    GROUP_NORMAL:  "everything else (well-behaved distribution)",
}


def _assign_group(p: dict) -> tuple:
    if p["is_port"]:
        return GROUP_PORT, "col name contains 'port' → hand-crafted bucket (0-6)"

    if p["is_binary"]:
        return GROUP_BINARY, f"unique⊆{{0,1}}, n_unique={p['n_unique']}"

    if p["is_low_card"]:
        return GROUP_LOWCARD, f"n_unique={p['n_unique']} < 20 → label-encode"

    tr, sr = p["tail_ratio"], p["spread_ratio"]
    lr     = p["left_ratio"]

    if tr > 5.0 or sr < 0.05 or lr > 0.8 or lr < 0.2:
        return (GROUP_SKEWED,
                f"tail_ratio={tr:.2f}, spread_ratio={sr:.4f}, left_ratio={lr:.3f} "
                f"→ piecewise sqrt / symmetric log")

    return (GROUP_NORMAL,
            f"well-behaved (tail_ratio={tr:.2f}, spread_ratio={sr:.3f}) → robust minmax")


_MIN_BULK_RES = 800


def _fits_token_budget(p: dict, scale_factor: int = DEFAULT_SCALE_FACTOR) -> bool:
    full_range = p["p100"] - p["p0"]
    n_unique   = p.get("n_unique", 0)
    return full_range <= scale_factor and n_unique <= scale_factor


_HEAVY_TAIL_RATIO = 2.0


def _has_heavy_tail(p: dict) -> bool:
    tail_span = p["p100"] - p["p99"]
    body_span = p["p99"]  - p["p1"]
    if body_span <= _EPS:
        return tail_span > _EPS
    return (tail_span / body_span) >= _HEAVY_TAIL_RATIO


def _body_resolution(tokens: np.ndarray) -> int:
    t = np.asarray(tokens).flatten()
    if t.size == 0:
        return 0
    lo, hi = np.percentile(t, 5), np.percentile(t, 95)
    mid = t[(t >= lo) & (t <= hi)]
    return int(np.unique(mid).size) if mid.size else 0


def _select_strategy_percentile(p: dict, scale_factor: int) -> str:
    p0, p99 = p["p0"], p["p99"]
    skew    = abs(p.get("skew", 0.0))
    tail    = p.get("tail_ratio", 0.0)
    bipolar = p0 < 0 and abs(p0) > 0.05 * (abs(p99) + _EPS)
    bulk_res = (p["p90"] - p["p10"]) / (p99 - p["p1"] + _EPS) * scale_factor
    if bulk_res >= _MIN_BULK_RES:
        return STRAT_LINEAR
    if bipolar or skew > 4.0 or tail > 6.0:
        return STRAT_CBRT
    return STRAT_PIECEWISE if _has_heavy_tail(p) else STRAT_LINEAR


def _select_continuous_strategy(p: dict, scale_factor: int = DEFAULT_SCALE_FACTOR,
                                col_train: np.ndarray = None) -> str:
    if p.get("feature") in FORCE_QUANTILE_FEATURES:
        return STRAT_QUANTILE

    if _fits_token_budget(p, scale_factor):
        return STRAT_LINEAR

    if col_train is None:
        return _select_strategy_percentile(p, scale_factor)

    ctr = np.asarray(col_train, dtype=float).reshape(-1, 1)

    lin_res = _body_resolution(
        _robust_linear_scale(ctr, ctr, scale_factor, lo_pct=0, hi_pct=100))
    if lin_res >= _MIN_BULK_RES:
        return STRAT_LINEAR

    heavy = _has_heavy_tail(p)
    pw_res = 0
    if heavy:
        pw_res = _body_resolution(_piecewise_sqrt_scale(ctr, ctr, scale_factor))
        if pw_res >= _MIN_BULK_RES:
            return STRAT_PIECEWISE
    cb_res = _body_resolution(
        _signed_power_scale(ctr, ctr, scale_factor, 1.0 / 3.0))
    if cb_res >= _MIN_BULK_RES:
        return STRAT_CBRT

    qt_res = _body_resolution(_quantile_scale(ctr, ctr, scale_factor))
    cands  = [(STRAT_LINEAR, lin_res), (STRAT_CBRT, cb_res), (STRAT_QUANTILE, qt_res)]
    if heavy:
        cands.append((STRAT_PIECEWISE, pw_res))
    best   = max(r for _, r in cands)
    pref   = {STRAT_LINEAR: 0, STRAT_PIECEWISE: 1, STRAT_CBRT: 2, STRAT_QUANTILE: 3}
    ok = [s for s, r in cands if r >= 0.95 * best]
    return min(ok, key=lambda s: pref[s])


def phase2_group(profiles: dict,
                 fixed_normal_features: list = None,
                 report_path: str = "feature_groups_report.json",
                 scale_factor: int = DEFAULT_SCALE_FACTOR,
                 df: pd.DataFrame = None,
                 idx_train: np.ndarray = None) -> tuple:
    print(f"\n[Phase 2] Grouping → '{report_path}' …")

    fixed_normal_set = set(fixed_normal_features) if fixed_normal_features else set()
    if fixed_normal_set:
        print(f"      ℹ️  {len(fixed_normal_set)} features forced → NORMAL")
    empirical = df is not None and idx_train is not None
    print(f"      Strategy: {'THỰC NGHIỆM (đo body-resolution trên train)' if empirical else 'phân vị (fallback)'}")

    groups     = {g: [] for g in _ALL_GROUPS}
    strategies = {}
    rows       = []

    for col, p in profiles.items():
        if col in fixed_normal_set:
            grp, reason = GROUP_NORMAL, "Manually forced to normal group"
        else:
            grp, reason = _assign_group(p)

        if grp in (GROUP_NORMAL, GROUP_SKEWED):
            col_train = (df.loc[idx_train, col].values.astype(float)
                         if empirical else None)
            strategy = _select_continuous_strategy(p, scale_factor, col_train)
        else:
            strategy = grp
        strategies[col] = strategy

        heavy = _has_heavy_tail(p)
        is_pw = (strategy == STRAT_PIECEWISE)
        if is_pw:
            s             = float(p["p99"])
            pw_split      = round(s, 6)
            pw_tail_after = round(float(s + np.sqrt(max(p["p100"] - s, 0.0))), 6)
        else:
            pw_split = pw_tail_after = None

        groups[grp].append(col)
        rows.append({
            "feature": col, "group": grp, "strategy": strategy,
            "heavy_tail": heavy, "piecewise": is_pw,
            "pw_split": pw_split, "pw_tail_after": pw_tail_after,
            "reason": reason,
            "scaler": _SCALER_LABELS.get(grp, "unknown"),
            **{k: p[k] for k in [
                "p0","p1","p5","p10","p25","p50","p75","p90","p95","p99","p100",
                "iqr","tail_ratio","tail_75_ratio","tail_90_ratio",
                "left_ratio","spread_ratio","zero_ratio","n_unique",
                "mean","std","skew","kurtosis","cv",
            ]},
        })

    export_feature_groups_report(rows, report_path)

    W = 68
    print(f"\n  {'─'*W}")
    print(f"  {'Feature Group Summary':^{W}}")
    print(f"  {'─'*W}")
    print(f"  {'Group':<12}  {'N':>4}  {'Criteria':<38}  Example")
    print(f"  {'─'*W}")
    for grp in _ALL_GROUPS:
        cols = groups[grp]
        if not cols:
            continue
        ex = ", ".join(cols[:2]) + ("…" if len(cols) > 2 else "")
        print(f"  {grp:<12}  {len(cols):>4}  {_GROUP_CRITERIA[grp]:<38}  {ex}")
    print(f"  {'─'*W}")
    print(f"\n  Report → {Path(report_path).resolve()}")
    return groups, strategies


_PORT_MAP = {0: 0, 22: 1, 53: 2, 80: 3, 443: 4}

def _port_to_token(port) -> int:
    try:
        port = int(port)
    except (TypeError, ValueError):
        return 0
    if port in _PORT_MAP:
        return _PORT_MAP[port]
    return 5 if port < 1024 else 6


def _robust_linear_scale(X_train, X_all, scale_factor,
                         lo_pct=1, hi_pct=99) -> np.ndarray:
    lo    = np.percentile(X_train, lo_pct, axis=0)
    hi    = np.percentile(X_train, hi_pct, axis=0)
    denom = hi - lo
    denom[denom < _EPS] = _EPS
    normed = (np.clip(X_all, lo, hi) - lo) / denom
    return np.clip(np.round(normed * scale_factor).astype(np.int64),
                   0, scale_factor)


def _piecewise_sqrt_scale(X_train, X_all, scale_factor,
                          split_pcts=None) -> np.ndarray:
    n_cols = X_train.shape[1]
    if split_pcts is None:
        split_pcts = [99] * n_cols

    result = np.zeros_like(X_all, dtype=np.float64)

    for j in range(n_cols):
        col_tr  = X_train[:, j].astype(np.float64)
        col_all = X_all[:,   j].astype(np.float64)

        s = float(np.percentile(col_tr, split_pcts[j]))
        l = float(np.min(col_tr))
        if s - l < _EPS:
            s = l + _EPS

        def _transform(c):
            t = c.copy()
            mask = t > s
            t[mask] = s + np.sqrt(t[mask] - s)
            return t

        tr_t  = _transform(col_tr)
        all_t = _transform(col_all)

        m     = float(np.max(tr_t))
        denom = m - l
        if denom < _EPS:
            denom = _EPS

        normed = (all_t - l) / denom
        result[:, j] = np.clip(normed * scale_factor, 0.0, scale_factor)

    return np.clip(np.round(result).astype(np.int64), 0, scale_factor)


def _signed_power_scale(X_train, X_all, scale_factor, power: float) -> np.ndarray:
    n_cols = X_train.shape[1]
    result = np.zeros_like(X_all, dtype=np.float64)

    for j in range(n_cols):
        col_tr  = X_train[:, j].astype(np.float64)
        col_all = X_all[:,   j].astype(np.float64)

        tr_t  = np.sign(col_tr)  * np.abs(col_tr)  ** power
        all_t = np.sign(col_all) * np.abs(col_all) ** power

        l, m  = float(tr_t.min()), float(tr_t.max())
        denom = m - l if (m - l) > _EPS else _EPS

        normed = (all_t - l) / denom
        result[:, j] = np.clip(normed * scale_factor, 0.0, scale_factor)

    return np.clip(np.round(result).astype(np.int64), 0, scale_factor)


def _quantile_scale(X_train, X_all, scale_factor) -> np.ndarray:
    n_cols = X_train.shape[1]
    result = np.zeros_like(X_all, dtype=np.float64)

    for j in range(n_cols):
        col_tr  = X_train[:, j].astype(np.float64).reshape(-1, 1)
        col_all = X_all[:,   j].astype(np.float64).reshape(-1, 1)

        n_q = int(min(1000, max(10, np.unique(col_tr).size)))
        qt  = QuantileTransformer(
            n_quantiles=n_q,
            output_distribution="uniform",
            subsample=1_000_000_000,
            random_state=_SPLIT_RANDOM_STATE,
        )
        qt.fit(col_tr)
        normed = qt.transform(col_all).flatten()
        result[:, j] = np.clip(normed * scale_factor, 0.0, scale_factor)

    return np.clip(np.round(result).astype(np.int64), 0, scale_factor)


_SCALER_LABELS = {
    GROUP_BINARY:  "keep {0,1}",
    GROUP_PORT:    "bucket map (0-6)",
    GROUP_LOWCARD: "factorize (ordinal)",
    GROUP_SKEWED:  "no clip → per-feat (piecewise/cbrt/quantile/minmax)",
    GROUP_NORMAL:  "no clip → per-feat (minmax/piecewise/cbrt)",
}


def phase3_scale_and_tokenize(
    df, features, groups, idx_train, profiles,
    scale_factor: int = DEFAULT_SCALE_FACTOR,
    strategies: dict = None,
) -> np.ndarray:
    print(f"\n[Phase 3] Scaling & tokenising  (scale_factor={scale_factor:,}) …")

    n, F     = len(df), len(features)
    feat_idx = {col: i for i, col in enumerate(features)}
    X_tokens = np.zeros((n, F), dtype=np.int64)

    for grp, cols in groups.items():
        if not cols:
            continue
        ci = [feat_idx[c] for c in cols]
        print(f"\n  [{grp:<12}] {len(cols):>3} cols  ({_SCALER_LABELS.get(grp,'?')})", end="")

        if grp == GROUP_PORT:
            for col, i in zip(cols, ci):
                X_tokens[:, i] = df[col].apply(_port_to_token).values
            print(f"  →  token ∈ {{0..{_PORT_VOCAB-1}}}")
            continue

        if grp == GROUP_BINARY:
            for col, i in zip(cols, ci):
                X_tokens[:, i] = df[col].astype(np.int64).values.clip(0, 1)
            print("  →  token ∈ {0, 1}")
            continue

        if grp == GROUP_LOWCARD:
            for col, i in zip(cols, ci):
                train_vals = sorted(df.loc[idx_train, col].dropna().unique())
                val_map    = {v: k for k, v in enumerate(train_vals)}
                oov_token  = len(train_vals)
                X_tokens[:, i] = (
                    df[col].map(val_map).fillna(oov_token)
                    .astype(np.int64).values
                )
            print("  →  token ∈ [0, n_unique]  (OOV = n_unique)")
            continue

        X_tr  = df.loc[idx_train, cols].values.astype(float)
        X_all = df[cols].values.astype(float)
        profs = [profiles[c] for c in cols]

        X_tr_c, X_all_c = X_tr, X_all

        tokens      = np.zeros((n, len(cols)), dtype=np.int64)
        strat_count = {}
        for k, (col, p) in enumerate(zip(cols, profs)):
            if strategies is not None and col in strategies:
                strat = strategies[col]
            else:
                strat = _select_continuous_strategy(
                    p, scale_factor, X_tr_c[:, k])
            strat_count[strat] = strat_count.get(strat, 0) + 1

            col_tr_2d  = X_tr_c[:,  k].reshape(-1, 1)
            col_all_2d = X_all_c[:, k].reshape(-1, 1)

            if strat == STRAT_CBRT:
                tk = _signed_power_scale(col_tr_2d, col_all_2d, scale_factor, 1.0 / 3.0)
            elif strat == STRAT_SQRT:
                tk = _signed_power_scale(col_tr_2d, col_all_2d, scale_factor, 1.0 / 2.0)
            elif strat == STRAT_PIECEWISE:
                tk = _piecewise_sqrt_scale(col_tr_2d, col_all_2d, scale_factor)
            elif strat == STRAT_QUANTILE:
                tk = _quantile_scale(col_tr_2d, col_all_2d, scale_factor)
            else:
                tk = _robust_linear_scale(col_tr_2d, col_all_2d, scale_factor,
                                          lo_pct=0, hi_pct=100)
            tokens[:, k] = tk.flatten()

        X_tokens[:, ci] = tokens
        strat_str = ", ".join(f"{s}×{c}" for s, c in sorted(strat_count.items()))
        print(f"  →  token ∈ [{int(tokens.min())}, {int(tokens.max())}]  "
              f"({strat_str})")

    vocab = int(X_tokens.max()) + 1
    print(f"\n  {'─'*52}")
    print(f"  Vocab size = {vocab:,}  → dùng làm NUM_BINS trong start.py")
    return X_tokens


def export_feature_groups_report(rows, report_path):
    """Thay cho tokenizer_report.py: ghi báo cáo nhóm/chiến lược từng cột ra JSON."""
    Path(report_path).write_text(json.dumps(rows, default=str, ensure_ascii=False, indent=1))

TOKENIZER_NS = types.SimpleNamespace(phase1_profile=phase1_profile, phase2_group=phase2_group,
                                     phase3_scale_and_tokenize=phase3_scale_and_tokenize)


# ======================================================================================
# TOKENIZER v2 - ghi đè cách chọn chiến lược và phase 3 (TOKENIZER_V2=False -> hành vi gốc)
#  1. STRAT_LOG = signed log1p, BẮT BUỘC cho Flow Duration / *IAT* / Active* / Idle*
#  2. bỏ chiến lược dồn > 5% dòng fit vào MỘT token chứa nhiều giá trị thô khác nhau
#  3. SPREAD_DISCRETE: binary / low_card trải ra 0..scale_factor để PLE nhìn thấy
# ======================================================================================
import re
import numpy as np
from sklearn.preprocessing import QuantileTransformer

TOKENIZER_V2          = True
SPREAD_DISCRETE       = TOKENIZER_V2
STRAT_LOG             = "signed_log1p"
_TIME_FEATURE_RE      = re.compile(r"\b(duration|iat|active|idle)\b", re.I)
_MAX_COLLISION_SHARE  = 0.05     # tối đa 5% dòng fit được phép dồn vào một token "nhiều giá trị"
_PREF = {STRAT_LINEAR: 0, STRAT_PIECEWISE: 1, STRAT_LOG: 2, STRAT_CBRT: 3, STRAT_QUANTILE: 4}


def _signed_log(a):
    return np.sign(a) * np.log1p(np.abs(a))


def _signed_log_scale(X_train, X_all, scale_factor) -> np.ndarray:
    tr = _signed_log(np.asarray(X_train, dtype=np.float64))
    al = _signed_log(np.asarray(X_all, dtype=np.float64))
    l, m = tr.min(axis=0), tr.max(axis=0)
    denom = np.where((m - l) > _EPS, m - l, _EPS)
    res = np.clip((al - l) / denom * scale_factor, 0.0, scale_factor)
    return np.clip(np.round(res).astype(np.int64), 0, scale_factor)


def _collision_share(values, tokens) -> float:
    """Tỉ lệ dòng nằm trong token đông nhất mà token đó gộp >1 giá trị thô khác nhau."""
    v = np.asarray(values, dtype=np.float64).ravel()
    t = np.asarray(tokens).ravel()
    if t.size == 0:
        return 0.0
    pairs = np.unique(np.stack([t.astype(np.float64), v], 1), axis=0)
    tok_u, n_vals = np.unique(pairs[:, 0], return_counts=True)
    multi = tok_u[n_vals > 1]
    if multi.size == 0:
        return 0.0
    rows = np.bincount(np.searchsorted(tok_u, t.astype(np.float64)), minlength=tok_u.size)
    return float(rows[np.isin(tok_u, multi)].max() / t.size)


def _is_time_feature(name) -> bool:
    return bool(_TIME_FEATURE_RE.search(str(name)))


def _select_continuous_strategy(p: dict, scale_factor: int = DEFAULT_SCALE_FACTOR,
                                col_train: np.ndarray = None) -> str:
    name = p.get("feature")
    if name in FORCE_QUANTILE_FEATURES:
        return STRAT_QUANTILE
    if TOKENIZER_V2 and _is_time_feature(name):
        return STRAT_LOG
    if _fits_token_budget(p, scale_factor):
        return STRAT_LINEAR
    if col_train is None:
        return _select_strategy_percentile(p, scale_factor)

    ctr = np.asarray(col_train, dtype=float).reshape(-1, 1)
    makers = {
        STRAT_LINEAR: lambda: _robust_linear_scale(ctr, ctr, scale_factor, lo_pct=0, hi_pct=100),
        STRAT_CBRT:   lambda: _signed_power_scale(ctr, ctr, scale_factor, 1.0 / 3.0),
        STRAT_QUANTILE: lambda: _quantile_scale(ctr, ctr, scale_factor),
    }
    if _has_heavy_tail(p):
        makers[STRAT_PIECEWISE] = lambda: _piecewise_sqrt_scale(ctr, ctr, scale_factor)
    if TOKENIZER_V2:
        makers[STRAT_LOG] = lambda: _signed_log_scale(ctr, ctr, scale_factor)

    cands = []
    for s, mk in makers.items():
        tk = mk()
        res = _body_resolution(tk)
        col = _collision_share(ctr, tk) if TOKENIZER_V2 else 0.0
        cands.append((s, res, col))
    ok = [c for c in cands if c[1] >= _MIN_BULK_RES and c[2] <= _MAX_COLLISION_SHARE]
    if ok:
        return min(ok, key=lambda c: _PREF[c[0]])[0]
    if not TOKENIZER_V2:                                   # hành vi gốc
        best = max(r for _, r, _ in cands)
        return min([s for s, r, _ in cands if r >= 0.95 * best], key=lambda s: _PREF[s])
    return min(cands, key=lambda c: (c[2], -c[1], _PREF[c[0]]))[0]   # ít va chạm nhất


def _discrete_scale(n_codes, scale_factor):
    """Hệ số trải token rời rạc: code k -> round(k * SF / n_codes). Không trải -> 1."""
    return (scale_factor / max(n_codes, 1)) if SPREAD_DISCRETE else 1.0


def phase3_scale_and_tokenize(df, features, groups, idx_train, profiles,
                              scale_factor: int = DEFAULT_SCALE_FACTOR, strategies: dict = None) -> np.ndarray:
    print(f"\n[Phase 3 v2] Scaling & tokenising  (scale_factor={scale_factor:,}, "
          f"SPREAD_DISCRETE={SPREAD_DISCRETE}) …")
    n, F = len(df), len(features)
    feat_idx = {col: i for i, col in enumerate(features)}
    X_tokens = np.zeros((n, F), dtype=np.int64)
    for grp, cols in groups.items():
        if not cols:
            continue
        ci = [feat_idx[c] for c in cols]
        print(f"\n  [{grp:<12}] {len(cols):>3} cols", end="")
        if grp == GROUP_PORT:
            for col, i in zip(cols, ci):
                X_tokens[:, i] = df[col].apply(_port_to_token).values
            continue
        if grp == GROUP_BINARY:
            k = _discrete_scale(1, scale_factor)
            for col, i in zip(cols, ci):
                X_tokens[:, i] = np.round(df[col].astype(np.int64).values.clip(0, 1) * k).astype(np.int64)
            print(f"  →  token ∈ {{0, {int(round(k))}}}")
            continue
        if grp == GROUP_LOWCARD:
            for col, i in zip(cols, ci):
                train_vals = sorted(df.loc[idx_train, col].dropna().unique())
                val_map = {v: k for k, v in enumerate(train_vals)}
                oov = len(train_vals)
                codes = df[col].map(val_map).fillna(oov).astype(np.int64).values
                X_tokens[:, i] = np.round(codes * _discrete_scale(oov, scale_factor)).astype(np.int64)
            print("  →  low_card (OOV = token lớn nhất)")
            continue
        X_tr = df.loc[idx_train, cols].values.astype(float)
        X_all = df[cols].values.astype(float)
        cnt = {}
        for k, (col, p) in enumerate(zip(cols, [profiles[c] for c in cols])):
            strat = strategies[col] if strategies and col in strategies else \
                _select_continuous_strategy(p, scale_factor, X_tr[:, k])
            cnt[strat] = cnt.get(strat, 0) + 1
            a, b = X_tr[:, k].reshape(-1, 1), X_all[:, k].reshape(-1, 1)
            if strat == STRAT_LOG:
                tk = _signed_log_scale(a, b, scale_factor)
            elif strat == STRAT_CBRT:
                tk = _signed_power_scale(a, b, scale_factor, 1.0 / 3.0)
            elif strat == STRAT_SQRT:
                tk = _signed_power_scale(a, b, scale_factor, 0.5)
            elif strat == STRAT_PIECEWISE:
                tk = _piecewise_sqrt_scale(a, b, scale_factor)
            elif strat == STRAT_QUANTILE:
                tk = _quantile_scale(a, b, scale_factor)
            else:
                tk = _robust_linear_scale(a, b, scale_factor, lo_pct=0, hi_pct=100)
            X_tokens[:, ci[k]] = tk.flatten()
        print("  →  " + ", ".join(f"{s}×{c}" for s, c in sorted(cnt.items())))
    print(f"\n  Vocab size = {int(X_tokens.max()) + 1:,}")
    return X_tokens


TOKENIZER_NS = types.SimpleNamespace(phase1_profile=phase1_profile, phase2_group=phase2_group,
                                     phase3_scale_and_tokenize=phase3_scale_and_tokenize)

print(f"Tokenizer v2 đã nạp: TOKENIZER_V2={TOKENIZER_V2}, SPREAD_DISCRETE={SPREAD_DISCRETE}")

## TOKENIZER TRIỂN KHAI (fit_tokenizer / transform_tokens - v2)
Thay cho HAI cell trùng nhau ở bản gốc. Phải giống hệt phase 3 v2 (cell XUẤT TRIỂN KHAI A kiểm tra lại).

In [ ]:
"""
Tokenizer cho Deep_MHA_Tabular, tách thành 2 bước fit / transform để triển khai.

Tái lập CHÍNH XÁC phase3_scale_and_tokenize() trong notebook (tabular_tokenizer.py):
  - Trong notebook, tham số (min/max, percentile, QuantileTransformer...) được tính lại mỗi lần
    gọi và KHÔNG được lưu. Module này tính chúng một lần trên tập fit (A_S0) và lưu vào dict
    `params`, để server áp dụng cho dữ liệu mới mà không cần dữ liệu train.
  - `params` chỉ gồm kiểu Python/numpy + đối tượng sklearn QuantileTransformer
    (không pickle class tự định nghĩa), nạp bằng joblib.load là dùng được.

Dùng ở 2 nơi, code GIỐNG HỆT nhau:
  - Notebook Kaggle (Cell A): fit_tokenizer(...) rồi kiểm tra transform_tokens(...) == X_tok gốc.
  - Server (api/app.py): transform_tokens(params, X_raw).
"""
import numpy as np
from sklearn.preprocessing import QuantileTransformer

_EPS = 1e-9
_SPLIT_RANDOM_STATE = 42
_PORT_MAP = {0: 0, 22: 1, 53: 2, 80: 3, 443: 4}

STRAT_LINEAR = "robust_linear"
STRAT_QUANTILE = "quantile_rank"
STRAT_SQRT = "signed_sqrt"
STRAT_CBRT = "signed_cbrt"
STRAT_PIECEWISE = "piecewise_sqrt"


def _as_raw(X):
    """Dữ liệu train được lưu float32 rồi đổi sang float64 trước khi token hóa -> làm y hệt."""
    return np.asarray(X, dtype=np.float64).astype(np.float32).astype(np.float64)


def _to_tokens(normed, sf):
    res = np.clip(normed * sf, 0.0, sf)
    return np.clip(np.round(res).astype(np.int64), 0, sf)


def _port_to_token(port):
    try:
        port = int(port)
    except (TypeError, ValueError):
        return 0
    if port in _PORT_MAP:
        return _PORT_MAP[port]
    return 5 if port < 1024 else 6


# ------------------------------------------------------------------ fit / transform (triển khai)
def fit_tokenizer(X_fit, features, groups, strategies, scale_factor):
    X_fit = _as_raw(X_fit)
    features = list(features)
    fidx = {c: i for i, c in enumerate(features)}
    group_of = {c: g for g, cols in groups.items() for c in cols}
    missing = [c for c in features if c not in group_of]
    assert not missing, f"Cột không thuộc nhóm nào: {missing}"
    cols = []
    for c in features:
        g = group_of[c]
        col = X_fit[:, fidx[c]]
        p = {"name": c, "group": g}
        if g == "port":
            p["kind"] = "port"
        elif g == "binary":
            p.update(kind="binary", spread=_discrete_scale(1, scale_factor))
        elif g == "low_card":
            vals = np.unique(col[~np.isnan(col)])
            p.update(kind="low_card", values=vals.tolist(), spread=_discrete_scale(len(vals), scale_factor))
        else:
            strat = strategies[c]
            p["kind"] = strat
            if strat == STRAT_LOG:
                t = _signed_log(col)
                l, m = float(t.min()), float(t.max())
                p.update(l=l, denom=(m - l) if (m - l) > _EPS else _EPS)
            elif strat in (STRAT_CBRT, STRAT_SQRT):
                power = 1.0 / 3.0 if strat == STRAT_CBRT else 0.5
                tr_t = np.sign(col) * np.abs(col) ** power
                l, m = float(tr_t.min()), float(tr_t.max())
                p.update(power=power, l=l, denom=(m - l) if (m - l) > _EPS else _EPS)
            elif strat == STRAT_PIECEWISE:
                s = float(np.percentile(col, 99)); l = float(np.min(col))
                if s - l < _EPS:
                    s = l + _EPS
                tr_t = col.copy(); mask = tr_t > s
                tr_t[mask] = s + np.sqrt(tr_t[mask] - s)
                denom = float(np.max(tr_t)) - l
                p.update(s=s, l=l, denom=denom if denom >= _EPS else _EPS)
            elif strat == STRAT_QUANTILE:
                n_q = int(min(1000, max(10, np.unique(col).size)))
                qt = QuantileTransformer(n_quantiles=n_q, output_distribution="uniform",
                                         subsample=1_000_000_000, random_state=_SPLIT_RANDOM_STATE)
                qt.fit(col.reshape(-1, 1)); p["qt"] = qt
            else:
                p["kind"] = STRAT_LINEAR
                lo, hi = float(np.percentile(col, 0)), float(np.percentile(col, 100))
                p.update(lo=lo, hi=hi, denom=(hi - lo) if (hi - lo) >= _EPS else _EPS)
        cols.append(p)
    return {"format": "ddos_tokenizer/v2", "features": features,
            "scale_factor": int(scale_factor), "columns": cols}


def transform_tokens(params, X):
    X = _as_raw(X)
    if X.ndim != 2 or X.shape[1] != len(params["features"]):
        raise ValueError(f"Cần mảng (n, {len(params['features'])})")
    sf = params["scale_factor"]
    out = np.zeros(X.shape, dtype=np.int64)
    for j, p in enumerate(params["columns"]):
        x = X[:, j]; k = p["kind"]
        if k == "port":
            out[:, j] = [_port_to_token(v) for v in x]
        elif k == "binary":
            out[:, j] = np.round(x.astype(np.int64).clip(0, 1) * p.get("spread", 1.0)).astype(np.int64)
        elif k == "low_card":
            vals = np.asarray(p["values"], dtype=np.float64); oov = len(vals)
            pos = np.searchsorted(vals, x); pos_c = np.minimum(pos, max(oov - 1, 0))
            hit = (vals[pos_c] == x) if oov else np.zeros(len(x), bool)
            out[:, j] = np.round(np.where(hit, pos_c, oov) * p.get("spread", 1.0)).astype(np.int64)
        elif k == STRAT_LOG:
            out[:, j] = _to_tokens((_signed_log(x) - p["l"]) / p["denom"], sf)
        elif k in (STRAT_CBRT, STRAT_SQRT):
            t = np.sign(x) * np.abs(x) ** p["power"]
            out[:, j] = _to_tokens((t - p["l"]) / p["denom"], sf)
        elif k == STRAT_PIECEWISE:
            t = x.copy(); mask = t > p["s"]
            t[mask] = p["s"] + np.sqrt(t[mask] - p["s"])
            out[:, j] = _to_tokens((t - p["l"]) / p["denom"], sf)
        elif k == STRAT_QUANTILE:
            out[:, j] = _to_tokens(p["qt"].transform(x.reshape(-1, 1)).flatten(), sf)
        elif k == STRAT_LINEAR:
            normed = (np.clip(x, p["lo"], p["hi"]) - p["lo"]) / p["denom"]
            out[:, j] = np.clip(np.round(normed * sf).astype(np.int64), 0, sf)
        else:
            raise ValueError(f"Chiến lược không hỗ trợ: {k}")
    return out


def describe(params):
    """Bản tóm tắt dạng JSON (không gồm QuantileTransformer) để đọc / đưa vào báo cáo."""
    rows = []
    for p in params["columns"]:
        r = {k: v for k, v in p.items() if k != "qt"}
        if "qt" in p:
            r["n_quantiles"] = int(p["qt"].n_quantiles_)
        rows.append(r)
    return {"format": params["format"], "scale_factor": params["scale_factor"], "columns": rows}

## GIAI ĐOẠN 8 - Tokenize (CHỈ fit trên train Stage 0), xuất data.npz / meta.json

In [ ]:
# ===== GIAI ĐOẠN 8 - Tokenize bằng tabular_tokenizer.py (và/hoặc z-score), CHỈ fit trên train Stage 0, xuất file =====
# ---------------------------------------------------------------------------------------------
# Tiền xử lý inf / NaN - dùng CHUNG cho GĐ8, XUẤT TRIỂN KHAI và server (lưu trong tokenizer.joblib)
# ---------------------------------------------------------------------------------------------
def build_preprocess(raw_features, flag_specs, X_fit_raw):
    """Median từng cột trên tập fit sau khi đổi ±inf -> NaN. flag_specs: [(tên cờ, cột gốc, loại)]."""
    X = np.array(X_fit_raw, dtype=np.float64)
    X[~np.isfinite(X)] = np.nan
    with np.errstate(all="ignore"):
        med = np.array([np.nanmedian(X[:, j]) if np.isfinite(X[:, j]).any() else 0.0 for j in range(X.shape[1])])
    return dict(raw_features=list(raw_features), flags=[list(f) for f in flag_specs],
                median={c: float(m) for c, m in zip(raw_features, med)})


def preprocess_raw(pp, X_raw):
    """X_raw: (n, len(raw_features)) đúng thứ tự pp['raw_features'], có thể chứa ±inf/NaN.
    Trả về ma trận cho tokenizer: [giá trị đã impute median | cờ 0/1 theo pp['flags']]."""
    X = np.array(X_raw, dtype=np.float64)
    col = {c: j for j, c in enumerate(pp["raw_features"])}
    F = np.zeros((len(X), len(pp["flags"])), dtype=np.float64)
    for k, (_, c, kind) in enumerate(pp["flags"]):
        v = X[:, col[c]]
        F[:, k] = np.isposinf(v) if kind == "posinf" else (np.isneginf(v) if kind == "neginf" else np.isnan(v))
    bad = ~np.isfinite(X)
    if bad.any():
        med = np.array([pp["median"][c] for c in pp["raw_features"]])
        X[bad] = med[np.where(bad)[1]]
    return np.hstack([X, F])


def model_features(pp):
    return list(pp["raw_features"]) + [f[0] for f in pp["flags"]]


def raw_to_tokens(params, raw):
    """raw: DataFrame có các cột đặc trưng THÔ (hoặc ndarray đúng thứ tự). Dùng cho server và kiểm tra."""
    pp = params.get("preprocess")
    cols = pp["raw_features"] if pp else params["features"]
    X = raw[cols].to_numpy(dtype=np.float64) if hasattr(raw, "columns") else np.asarray(raw, dtype=np.float64)
    return transform_tokens(params, preprocess_raw(pp, X) if pp else X)


if not RUN_PREPROCESS:
    print('Bỏ qua (RUN_PREPROCESS=False)')
else:
    import sys, types

    split_id = df["split"].map({"train": 0, "val": 1, "test": 2}).to_numpy(dtype=np.int8)
    day_id   = df["day"].map({"day1": 1, "day2": 2}).to_numpy(dtype=np.int8)

    if SCALER_FIT == "A_S0":
        fit_mask = (A_stage == 0) & (split_id == 0) & A_use
    else:
        fit_mask = (day_id == 1) & (split_id == 0)
    assert fit_mask.sum() > 0, "Tập fit rỗng - kiểm tra stage 0."
    idx_train = np.where(fit_mask)[0]
    print(f"Tập FIT ({SCALER_FIT}): {len(idx_train):,} dòng "
          f"-> các lớp: {sorted(df.loc[fit_mask, 'label'].unique())}")

    # 8-0. inf/NaN: cờ + impute median (median CHỈ tính trên tập fit) -> ma trận đầu vào tokenizer
    X_raw_ = df[feats].to_numpy(dtype="float64")
    if INF_NAN_MODE == "flag_impute":
        PP = build_preprocess(feats, INF_FLAGS, X_raw_[idx_train])
        X_model = preprocess_raw(PP, X_raw_)
        model_feats = model_features(PP)
        _imp = {c: PP["median"][c] for c in feats if (~np.isfinite(X_raw_[:, feats.index(c)])).any()}
        print(f"8-0. Impute median (tập fit) cho {len(_imp)} cột: {_imp}")
        print(f"     Đầu vào model: {len(feats)} đặc trưng + {len(PP['flags'])} cờ = {len(model_feats)} cột")
    else:
        PP, X_model, model_feats = None, X_raw_, list(feats)
    del X_raw_
    assert np.isfinite(X_model).all(), "Còn inf/NaN sau tiền xử lý"

    out = dict(y=df["y"].to_numpy(), day=day_id, split=split_id,
               A_stage=A_stage, A_use=A_use, B_stage=B_stage, ts=df["ts"].to_numpy())
    meta = dict(features=model_feats, raw_features=feats, preprocess=PP, classes=classes, label2id=label2id, stages_A=STAGES_A,
                day_patterns=DAY_PATTERNS, scaler_fit=SCALER_FIT, tokenizer_mode=TOKENIZER_MODE,
                params=dict(SEED=SEED, CAP_PER_DAY_LABEL=CAP_PER_DAY_LABEL, SPLIT_FRAC=SPLIT_FRAC,
                            GAP_S=GAP_S, PRE_CAP_ATTACK=PRE_CAP_ATTACK, DEDUP_VECTORS=DEDUP_VECTORS,
                            DEDUP_MODE=DEDUP_MODE, INF_NAN_MODE=INF_NAN_MODE, PURGE_EXACT_DUP=PURGE_EXACT_DUP,
                            ATTACK_WINDOWS={f"{k[0]}/{k[1]}": v for k, v in ATTACK_WINDOWS.items()}),
                shared_classes_B=sorted(shared))

    # ---------------- 8a. z-score kiểu cũ (float) ----------------
    if TOKENIZER_MODE in ("zscore", "both"):
        X_log = signed_log(X_model)
        mu, sd = X_log[fit_mask].mean(0), X_log[fit_mask].std(0)
        sd[sd < 1e-6] = 1.0
        out["X"] = np.clip((X_log - mu) / sd, -10, 10).astype("float32")
        np.savez(PROCESSED / "scaler.npz", mu=mu, sd=sd)
        del X_log
        print("8a. Đã tạo X (z-score), shape", out["X"].shape)

    # ---------------- 8b. token nguyên bằng tabular_tokenizer.py của bạn ----------------
    if TOKENIZER_MODE in ("ple_tokens", "both"):
        tt = TOKENIZER_NS
        SF = TOKEN_SCALE_FACTOR
        df_tok = pd.DataFrame(X_model, columns=model_feats)   # RangeIndex: idx_train là vị trí dòng
        profiles = tt.phase1_profile(df_tok, model_feats, idx_train)
        groups, strategies = tt.phase2_group(
            profiles, fixed_normal_features=None,
            report_path=str(PROCESSED / "feature_groups_report.json"),
            scale_factor=SF, df=df_tok, idx_train=idx_train)
        X_tok = tt.phase3_scale_and_tokenize(df_tok, model_feats, groups, idx_train, profiles, SF, strategies=strategies)
        del df_tok
        assert X_tok.min() >= 0 and X_tok.max() <= 32767, "token vượt int16"
        if groups.get("port"):
            print("CẢNH BÁO: có cột bị xếp nhóm 'port' (tên chứa 'port'):", groups["port"])

        # hằng số chuẩn hoá theo TỪNG cột (chỉ suy từ tập fit, không nhìn dữ liệu tương lai)
        group_of = {c: g for g, cols in groups.items() for c in cols}
        feat_max = np.ones(len(model_feats), dtype=np.float32)
        for j, c in enumerate(model_feats):
            g = group_of[c]
            feat_max[j] = (SF if (g in ("normal", "skewed") or
                                 (SPREAD_DISCRETE and g in ("binary", "low_card"))) else
                           6 if g == "port" else
                           1 if g == "binary" else
                           max(1, profiles[c]["n_unique"]))          # low_card: OOV = n_unique
        out["X_tok"] = X_tok.astype(np.int16)
        meta["tokenizer"] = dict(scale_factor=SF, num_bins=SF + 1, groups=groups, strategies=strategies,
                                 feat_max=feat_max.tolist(),
                                 n_tokens_max=int(X_tok.max()),
                                 group_counts={g: len(c) for g, c in groups.items()})
        print("8b. Đã tạo X_tok (token int16), shape", X_tok.shape, "| max token =", int(X_tok.max()))
        print("    Số cột theo nhóm:", meta["tokenizer"]["group_counts"])
        from collections import Counter
        print("    Chiến lược:", dict(Counter(strategies.values())))

    np.savez_compressed(PROCESSED / "data.npz", **out)
    (PROCESSED / "meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
    del X_model; gc.collect()
    print("Đã lưu:", [p.name for p in PROCESSED.iterdir()])

## XUẤT TRIỂN KHAI (A) - Lưu tokenizer đã fit + tập test THÔ  (bật bằng RUN_DEPLOY_EXPORT)

In [ ]:
# ===== XUẤT TRIỂN KHAI (A) - Lưu tokenizer đã fit + tập test THÔ =====
if not (RUN_DEPLOY_EXPORT and RUN_PREPROCESS):
    print("Bỏ qua XUẤT TRIỂN KHAI (A) (RUN_DEPLOY_EXPORT=False hoặc RUN_PREPROCESS=False)")
else:
    import sys, platform, joblib, sklearn

    EXPORT = WORK_DIR / "deploy_export"
    EXPORT.mkdir(parents=True, exist_ok=True)

    # 1) Fit tokenizer trên đúng tập fit A_S0 (giống hệt Giai đoạn 8)
    X_raw_all = df[feats].to_numpy(dtype="float64")                       # giá trị THÔ (có thể có inf/NaN)
    X_raw_all = preprocess_raw(PP, X_raw_all) if PP else X_raw_all        # cờ + impute median giống GĐ8
    tok_params = fit_tokenizer(X_raw_all[idx_train], model_feats, groups, strategies, TOKEN_SCALE_FACTOR)
    if PP:
        tok_params["preprocess"] = PP          # server: raw_to_tokens(tok_params, bảng_thô) tự dựng cờ + impute

    # 2) KIỂM TRA: transform toàn bộ dữ liệu phải ra ĐÚNG token X_tok mà model đã học
    X_chk = transform_tokens(tok_params, X_raw_all)
    X_ref = out["X_tok"].astype(np.int64)
    n_bad_rows = int((X_chk != X_ref).any(axis=1).sum())
    print(f"Kiểm tra tokenizer: {n_bad_rows:,} / {len(X_ref):,} dòng lệch")
    assert n_bad_rows == 0, "Tokenizer xuất ra KHÔNG khớp X_tok - dừng lại, báo lỗi"
    _n = min(len(df), 20_000)                  # kiểm tra luôn đường đi từ bảng THÔ (như server nhận)
    assert (raw_to_tokens(tok_params, df.iloc[:_n]) == X_ref[:_n]).all(), "raw_to_tokens lệch X_tok"
    print("Bảng thô (có inf/NaN) -> raw_to_tokens -> token: KHỚP")
    del X_chk

    joblib.dump(tok_params, EXPORT / "tokenizer.joblib")
    (EXPORT / "tokenizer_summary.json").write_text(
        json.dumps(describe(tok_params), indent=1, ensure_ascii=False, default=float), encoding="utf-8")

    # 3) Tập test THÔ (đã làm sạch, CHƯA token hóa), đúng thứ tự dòng của select_view("A")
    test_A = A_use & (split_id == 2)
    cols_out = feats + ["label", "day"]
    (df.loc[test_A, cols_out].rename(columns={"label": "Label"})
       .to_csv(EXPORT / "test_viewA_raw.csv", index=False))
    print("test view A:", int(test_A.sum()), "dòng")

    # 4) (Cho demo drift) test ngày 2 của các lớp dùng chung hai ngày - giống view "cross_day"
    test_day2 = df["label"].isin(shared).to_numpy() & (day_id == 2) & (split_id == 2)
    (df.loc[test_day2, cols_out].rename(columns={"label": "Label"})
       .to_csv(EXPORT / "test_crossday_day2_raw.csv", index=False))
    print("test ngày 2 (cross_day):", int(test_day2.sum()), "dòng")

    # 5) Version thư viện (để Docker trên VM khớp)
    try:
        import torch
        torch_v = torch.__version__
    except ImportError:
        torch_v = None
    versions = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                    sklearn=sklearn.__version__, torch=torch_v)
    (EXPORT / "versions.json").write_text(json.dumps(versions, indent=2))
    print("Version:", versions)
    print("Đã lưu:", sorted(p.name for p in EXPORT.iterdir()))

## GIAI ĐOẠN 9 - Thống kê cho báo cáo và kiểm tra nhanh

In [ ]:
# ===== GIAI ĐOẠN 9 - Thống kê cho báo cáo và kiểm tra nhanh (rò rỉ, khoảng cách giữa hai ngày) =====
if not (RUN_PREPROCESS and RUN_SANITY):
    print('Bỏ qua thống kê/kiểm tra nhanh')
else:
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import f1_score

    # 9a. Bảng số mẫu mỗi lớp x ngày x split (đưa vào chương dữ liệu)
    stats = df.groupby(["label", "day", "split"]).size().unstack([1, 2]).fillna(0).astype(int)
    stats.to_csv(PROCESSED / "stats_class_day_split.csv")
    print(stats)

    X_eval = out["X"] if "X" in out else out["X_tok"].astype("float32")   # cho cây/RF: token nguyên vẫn dùng được
    X = X_eval
    y = df["y"].to_numpy()
    tr = split_id == 0
    te = split_id == 2
    sub = np.where(tr)[0]
    sub = rng.choice(sub, min(len(sub), 100_000), replace=False)

    # 9b. "Đường tắt": cây quyết định độ sâu 3 trên TỪNG đặc trưng riêng lẻ.
    # Nếu một đặc trưng đơn lẻ cho Macro-F1 rất cao -> nghi ngờ rò rỉ / đặc trưng mang dấu vết thu thập.
    Xs, ys, Xte, yte = X[sub], y[sub], X[te], y[te]
    res = []
    for j, c in enumerate(meta["features"]):
        clf = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(Xs[:, [j]], ys)
        res.append((c, f1_score(yte, clf.predict(Xte[:, [j]]), average="macro")))
    res = sorted(res, key=lambda r: -r[1])[:8]
    print("\nTop đặc trưng đơn lẻ theo Macro-F1 trên test (cao bất thường = đáng ngờ):")
    for c, s in res:
        print(f"  {c:<35} {s:.3f}")

    # 9c. Xem trước khoảng cách giữa hai ngày: train ngày 1 -> test ngày 1 và test ngày 2 (các lớp dùng chung)
    sh_ids = [label2id[c] for c in shared]
    m_tr = tr & (day_id == 1) & np.isin(y, sh_ids)
    idx = np.where(m_tr)[0]
    if len(idx) == 0 or not (day_id == 2).any():
        print("Bỏ qua 9c: thiếu dữ liệu ngày 1 hoặc ngày 2 (kiểm tra Giai đoạn 1).")
    else:
        idx = rng.choice(idx, min(len(idx), 200_000), replace=False)
        rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, class_weight="balanced_subsample",
                                    random_state=SEED).fit(X[idx], y[idx])
        for d in (1, 2):
            mt = te & (day_id == d) & np.isin(y, sh_ids)
            if mt.sum():
                print(f"RF huấn luyện ngày 1 -> test ngày {d}: Macro-F1 = "
                      f"{f1_score(y[mt], rf.predict(X[mt]), average='macro'):.3f}  ({mt.sum():,} mẫu)")
        print("Khoảng cách giữa hai con số trên chính là mức drift thật của dữ liệu.")

## MÔ HÌNH + TRAINER (start.py - giữ nguyên kiến trúc)


In [ ]:
# ======================================================================================
# MÔ HÌNH + TRAINER: sao chép NGUYÊN VĂN từ start.py của bạn (không sửa kiến trúc).
# Chỉ bỏ phần `if __name__ == "__main__"` (thay bằng các cell chạy pipeline bên dưới).
# ======================================================================================
import os
import math
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.utils import check_random_state
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

# MLflow — tuỳ chọn: log params/metrics/artifacts. Không cài cũng chạy bình thường.
try:
    import mlflow
    _HAS_MLFLOW = True
except ImportError:
    _HAS_MLFLOW = False
    print("ℹ️  Không có mlflow — bỏ qua logging (pip install mlflow để bật).")


class FocalLoss(nn.Module):
    """
    Focal Loss — không dùng class_weights trực tiếp để tránh double-counting
    với focal modulation factor (1-pt)^gamma.
    Class imbalance được xử lý qua sample_weight bên ngoài nếu cần.
    """
    def __init__(self, gamma=2.0, reduction='mean', label_smoothing=0.0):
        super(FocalLoss, self).__init__()
        self.gamma     = gamma
        self.reduction = reduction
        # Không nhân class_weights ở đây — focal đã điều chỉnh theo pt
        self.ce = nn.CrossEntropyLoss(reduction='none', label_smoothing=label_smoothing)

    def forward(self, inputs, targets):
        ce_loss    = self.ce(inputs, targets)
        pt         = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss.sum()


# ==========================================
# 1. TABULAR EMBEDDING & MULTI-HEAD ATTENTION COMPONENTS
# ==========================================

class PLEValueEmbedding(nn.Module):
    """
    Piecewise Linear Encoding (PLE) — Monotonic value embedding.
    """
    def __init__(self, num_bins: int, value_dim: int, n_segments: int = None):
        super().__init__()
        self.num_bins   = max(num_bins, 1)
        self.value_dim  = value_dim
        # Clamp n_segments <= value_dim để tránh nén thông tin PLE
        raw_segments    = n_segments if n_segments is not None else value_dim
        self.n_segments = min(raw_segments, value_dim)
        self.proj = nn.Linear(self.n_segments, value_dim, bias=True)
        nn.init.xavier_uniform_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def _ple_encode(self, t: torch.Tensor) -> torch.Tensor:
        T    = self.n_segments
        w    = 1.0 / T
        left = torch.arange(T, dtype=torch.float32, device=t.device) * w
        fill = ((t.unsqueeze(-1) - left) / w).clamp(0.0, 1.0)
        return fill

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        t   = token_ids.float() / (self.num_bins - 1)
        ple = self._ple_encode(t)
        return self.proj(ple)


class SimpleValueEmbedding(nn.Module):
    """
    Embedding số ĐƠN GIẢN (baseline so với PLE): t∈[0,1] → MLP 2 lớp → value_dim.
    Không có cấu trúc piecewise; chỉ là MLP nhỏ trên giá trị chuẩn hoá.
    """
    def __init__(self, num_bins: int, value_dim: int, n_segments: int = None):
        super().__init__()
        self.num_bins = max(num_bins, 1)
        self.net = nn.Sequential(
            nn.Linear(1, value_dim),
            nn.GELU(),
            nn.Linear(value_dim, value_dim),
        )

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        t = (token_ids.float() / (self.num_bins - 1)).unsqueeze(-1)
        return self.net(t)


class MonotonicTabularEmbedding(nn.Module):
    def __init__(self, num_features: int, num_bins: int,
                 value_dim: int, feat_dim: int, n_segments: int = None,
                 value_embed: str = "ple"):
        super().__init__()
        self.num_features = num_features
        enc = PLEValueEmbedding if value_embed == "ple" else SimpleValueEmbedding
        self.value_embedding   = enc(num_bins, value_dim, n_segments)
        self.feature_embedding = nn.Embedding(num_features, feat_dim)
        nn.init.trunc_normal_(self.feature_embedding.weight, std=0.02)

    def forward(self, x_tokens: torch.Tensor) -> torch.Tensor:
        val_emb  = self.value_embedding(x_tokens)
        feat_idx = torch.arange(self.num_features, device=x_tokens.device)
        feat_emb = self.feature_embedding(feat_idx).unsqueeze(0)
        feat_emb = feat_emb.expand(x_tokens.size(0), -1, -1)
        return torch.cat([val_emb, feat_emb], dim=-1)


class ScaledDotProductMultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0, (
            f"embed_dim={embed_dim} phải chia hết cho num_heads={num_heads}"
        )
        self.num_heads = num_heads
        self.d_k       = embed_dim // num_heads
        self.d_model   = embed_dim

        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=False)
        self.attn_drop = nn.Dropout(dropout)

        for w in [self.W_q, self.W_k, self.W_v, self.W_o]:
            nn.init.xavier_uniform_(w.weight)

    def forward(self, x):
        N, S, _ = x.size()
        H, dk   = self.num_heads, self.d_k

        Q = self.W_q(x).view(N, S, H, dk).transpose(1, 2)
        K = self.W_k(x).view(N, S, H, dk).transpose(1, 2)
        V = self.W_v(x).view(N, S, H, dk).transpose(1, 2)

        scores  = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(dk)
        weights = self.attn_drop(torch.softmax(scores, dim=-1))
        ctx     = torch.matmul(weights, V)

        ctx = ctx.transpose(1, 2).contiguous().view(N, S, self.d_model)
        return self.W_o(ctx)


class ParallelBranchAttention(nn.Module):
    """
    Parallel Multi-Branch Attention (KHÔNG phải MoE — không có router).
      - num_branches nhánh xử lý SONG SONG, mỗi nhánh là 1 Multi-Head Attention
        độc lập với num_heads_per_branch head (W_q/W_k/W_v/W_o riêng).
      - Mặc định 4 nhánh × 4 head = 16 head.
      - Output = trung bình của các nhánh.
    """
    def __init__(self, embed_dim, num_branches=4, num_heads_per_branch=4, dropout=0.1):
        super().__init__()
        self.num_branches = num_branches
        self.branches = nn.ModuleList([
            ScaledDotProductMultiHeadAttention(embed_dim, num_heads_per_branch, dropout)
            for _ in range(num_branches)
        ])

    def forward(self, x):
        out = sum(branch(x) for branch in self.branches)
        return out / self.num_branches


class ParallelBranchTransformerBlock(nn.Module):
    """Khối Transformer Pre-LayerNorm dùng Parallel Multi-Branch Attention + FFN."""
    def __init__(self, embed_dim, num_branches, num_heads_per_branch, mlp_dropout):
        super().__init__()
        self.attn_norm = nn.LayerNorm(embed_dim)
        self.attention = ParallelBranchAttention(
            embed_dim, num_branches=num_branches,
            num_heads_per_branch=num_heads_per_branch, dropout=0.1)
        self.ffn_norm  = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(mlp_dropout),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        x = x + self.attention(self.attn_norm(x))
        x = x + self.ffn(self.ffn_norm(x))
        return x


class Deep_MHA_Tabular(nn.Module):
    """
    Mô hình chính: nhúng PLE + feature-identity → N tầng Transformer MHA →
    pooling 3 chiều (attention + max + mean) → MLP classifier.
    """
    def __init__(self, num_numeric_features, num_bins,
                 value_dim=224, feat_dim=64, n_segments=None,
                 num_layers=4, num_branches=4, num_heads_per_branch=4,
                 num_classes=5, mlp_dropout=0.4, value_embed="ple"):
        super().__init__()
        embed_dim = value_dim + feat_dim
        assert embed_dim % num_heads_per_branch == 0, (
            f"embed_dim={embed_dim} (value_dim={value_dim} + feat_dim={feat_dim}) "
            f"phải chia hết cho num_heads_per_branch={num_heads_per_branch}"
        )

        self.feature_embedder = MonotonicTabularEmbedding(
            num_features=num_numeric_features,
            num_bins=num_bins,
            value_dim=value_dim,
            feat_dim=feat_dim,
            n_segments=n_segments,
            value_embed=value_embed,
        )
        self.input_norm = nn.LayerNorm(embed_dim)
        self.layers = nn.ModuleList([
            ParallelBranchTransformerBlock(embed_dim, num_branches,
                                           num_heads_per_branch, mlp_dropout)
            for _ in range(num_layers)
        ])
        self.final_norm  = nn.LayerNorm(embed_dim)
        self.attn_pool_w = nn.Linear(embed_dim, 1)

        # Classifier nhận embed_dim * 3 vì pooling gồm attn + max + mean
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 3, embed_dim),
            nn.GELU(),
            nn.Dropout(mlp_dropout),
            nn.Linear(embed_dim, embed_dim // 2),
            nn.GELU(),
            nn.Dropout(mlp_dropout / 2),
            nn.Linear(embed_dim // 2, num_classes)
        )

    def forward(self, tokenized_x):
        x = self.feature_embedder(tokenized_x)
        x = self.input_norm(x)

        for layer in self.layers:
            x = layer(x)

        x = self.final_norm(x)

        # Pooling 3 chiều: attention + max + mean
        attn_w        = torch.softmax(self.attn_pool_w(x), dim=1)
        attn_pooled   = (x * attn_w).sum(dim=1)
        max_pooled, _ = x.max(dim=1)
        mean_pooled   = x.mean(dim=1)

        pooled = torch.cat([attn_pooled, max_pooled, mean_pooled], dim=1)
        return self.classifier(pooled)


# ==========================================
# 3. TRAINER COMPONENTS
# ==========================================
class Trainer:
    def __init__(self, model, criterion, optimizer, scheduler=None, device="cpu",
                 scheduler_per_batch=False, grad_clip=1.0):
        self.model               = model
        self.criterion           = criterion
        self.optimizer           = optimizer
        self.scheduler           = scheduler
        self.device              = device
        self.scheduler_per_batch = scheduler_per_batch
        self.grad_clip           = grad_clip
        self.best_val_f1         = 0.0
        self.best_model_state    = None
        self.scaler = (
            torch.amp.GradScaler('cuda')
            if getattr(device, 'type', 'cpu') == 'cuda'
            else None
        )

    def _run_epoch(self, loader, train=True):
        self.model.train(train)
        total_loss = 0.0
        preds, trues = [], []
        correct = 0
        total = 0
        ctx = torch.enable_grad() if train else torch.no_grad()

        pbar = tqdm(
            loader, total=len(loader),
            desc="Train" if train else "Eval",
            dynamic_ncols=True, leave=True, ascii=True
        )

        with ctx:
            for step, (num_x, lbl) in enumerate(pbar):
                num_x = num_x.to(torch.long, non_blocking=True).to(self.device)
                lbl   = lbl.to(torch.long,   non_blocking=True).to(self.device)

                if train:
                    self.optimizer.zero_grad(set_to_none=True)

                device_type = getattr(self.device, 'type', 'cpu')
                with torch.autocast(
                    device_type=device_type,
                    dtype=torch.float16,
                    enabled=(device_type == 'cuda')
                ):
                    logits = self.model(num_x)
                    loss   = self.criterion(logits, lbl)

                if train:
                    if self.scaler is not None:
                        self.scaler.scale(loss).backward()
                        if self.grad_clip is not None:
                            self.scaler.unscale_(self.optimizer)
                            torch.nn.utils.clip_grad_norm_(
                                self.model.parameters(), max_norm=self.grad_clip)
                        self.scaler.step(self.optimizer)
                        self.scaler.update()
                    else:
                        loss.backward()
                        if self.grad_clip is not None:
                            torch.nn.utils.clip_grad_norm_(
                                self.model.parameters(), max_norm=self.grad_clip)
                        self.optimizer.step()

                    if self.scheduler is not None and self.scheduler_per_batch:
                        self.scheduler.step()

                loss_val = loss.item()
                total_loss += loss_val
                pred = logits.argmax(1)
                correct += (pred == lbl).sum().item()
                total += lbl.size(0)
                preds.extend(pred.cpu().numpy())
                trues.extend(lbl.cpu().numpy())

                if step % 10 == 0:
                    current_acc = correct / max(total, 1)
                    if train:
                        current_lr = self.optimizer.param_groups[0]['lr']
                        pbar.set_postfix({
                            'loss': f'{loss_val:.4f}',
                            'acc':  f'{current_acc:.4f}',
                            'lr':   f'{current_lr:.2e}'
                        })
                    else:
                        pbar.set_postfix({
                            'loss': f'{loss_val:.4f}',
                            'acc':  f'{current_acc:.4f}'
                        })

        avg_loss = total_loss / len(loader)
        acc  = accuracy_score(trues, preds)
        f1_w = f1_score(trues, preds, average='weighted', zero_division=0)
        f1_m = f1_score(trues, preds, average='macro',    zero_division=0)
        return avg_loss, acc, f1_w, f1_m, preds, trues

    def fit(self, train_loader, val_loader, epochs=50, patience=18):
        print(f"\n{'='*80}")
        print(f"{'BẮT ĐẦU HUẤN LUYỆN':^80}")
        print(f"{'='*80}\n")

        no_improve = 0
        for epoch in range(1, epochs + 1):
            t0 = time.time()
            tr_loss, tr_acc, tr_f1w, tr_f1m, _, _ = self._run_epoch(train_loader, train=True)
            va_loss, va_acc, va_f1w, va_f1m, _, _ = self._run_epoch(val_loader,   train=False)

            if self.scheduler and not self.scheduler_per_batch:
                self.scheduler.step(va_loss)

            elapsed = time.time() - t0
            mark    = " ✦" if va_f1m > self.best_val_f1 else ""
            print(
                f"Epoch {epoch:03d}/{epochs} | "
                f"Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.4f} | "
                f"Val Loss: {va_loss:.4f} | Val Acc: {va_acc:.4f} | "
                f"Val F1-w: {va_f1w:.4f} | Val F1-m: {va_f1m:.4f} | "
                f"{elapsed:.1f}s{mark}"
            )

            if _HAS_MLFLOW and mlflow.active_run() is not None:
                mlflow.log_metrics({
                    "train_loss": tr_loss, "train_acc": tr_acc, "train_f1m": tr_f1m,
                    "val_loss":   va_loss, "val_acc":   va_acc,
                    "val_f1w":    va_f1w,  "val_f1m":   va_f1m,
                    "lr":         self.optimizer.param_groups[0]["lr"],
                }, step=epoch)

            if va_f1m > self.best_val_f1:
                self.best_val_f1      = va_f1m
                self.best_model_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"\n⚠️ Early stopping tại epoch {epoch}")
                    break

        print(f"\n✓ Nạp lại best checkpoint (Macro-F1 = {self.best_val_f1:.4f})")
        if self.best_model_state is not None:
            self.model.load_state_dict(self.best_model_state)
            self.model.to(self.device)

    def evaluate(self, val_loader, target_names=None):
        print("\nĐang đánh giá mô hình cuối cùng...")
        _, acc, f1w, f1m, preds, trues = self._run_epoch(val_loader, train=False)
        print("\nClassification Report:")
        print(classification_report(trues, preds, target_names=target_names, zero_division=0))
        print("Confusion Matrix:")
        print(confusion_matrix(trues, preds))
        print(f"\nAccuracy: {acc:.4f} | Weighted F1: {f1w:.4f} | Macro F1: {f1m:.4f}")
        return acc, f1m

    def export_misclassified_to_txt(self, loader, target_names, filepath="misclassified_samples.txt"):
        print(f"\nĐang trích xuất các mẫu dự đoán sai ra file: {filepath} ...")
        self.model.eval()
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write("True_Label\tPredicted_Label\tFeatures_Tokenized\n")
            with torch.no_grad():
                for num_x, lbl in loader:
                    num_x_device = num_x.to(torch.long).to(self.device)
                    lbl_device   = lbl.to(torch.long).to(self.device)
                    logits       = self.model(num_x_device)
                    preds        = logits.argmax(1).cpu().numpy()
                    trues        = lbl_device.cpu().numpy()
                    features     = num_x.numpy()
                    for i in range(len(preds)):
                        if preds[i] != trues[i]:
                            true_str = target_names[trues[i]]
                            pred_str = target_names[preds[i]]
                            feat_str = ",".join(map(str, features[i]))
                            f.write(f"{true_str}\t{pred_str}\t{feat_str}\n")
        print(f"✅ Đã ghi xong danh sách dự đoán sai vào: {filepath}")

## TIỆN ÍCH: chọn dữ liệu theo stage, replay buffer, metric (il_utils)


In [ ]:
import json
from pathlib import Path

import numpy as np
from sklearn.metrics import f1_score, recall_score

SPLIT_CODE = {"train": 0, "val": 1, "test": 2}


# ----------------------------------------------------------------------------- nạp dữ liệu
def load_processed(path):
    path = Path(path)
    d = np.load(path / "data.npz")
    M = json.loads((path / "meta.json").read_text())
    D = {k: d[k] for k in d.files}
    if "X_tok" not in D:
        raise RuntimeError("data.npz không có X_tok: chạy lại notebook tiền xử lý với "
                           "TOKENIZER_MODE='ple_tokens' hoặc 'both'.")
    D["F"] = D["X_tok"]                      # token nguyên (int16) cho Deep_MHA_Tabular
    return D, M


# ----------------------------------------------------------------------------- kịch bản A
def class_ids_by_stage(M):
    """Trả về (stages, benign_id): stages[t] = danh sách id lớp MỚI của stage t (chỉ lớp có trong dữ liệu)."""
    l2i = M["label2id"]
    stages = [[l2i[c] for c in st if c in l2i] for st in M["stages_A"]]
    return stages, l2i["Benign"]


def seen_ids(stages, benign, t):
    ids = {benign}
    for s in stages[: t + 1]:
        ids.update(s)
    ids = sorted(ids)
    if ids != list(range(len(ids))):
        raise RuntimeError(f"Id lớp đã thấy ở stage {t} không liên tục: {ids}. "
                           "Kiểm tra thứ tự lớp trong meta.json (phải theo thứ tự stage).")
    return ids


def select_A(D, t, split, upto=False):
    """
    split='train'/'val': lớp mới của stage t (+ phần benign của stage t); upto=True: gộp mọi stage 0..t (cho joint retrain).
    split='test'       : toàn bộ lớp đã thấy (stage <= t) + benign test.
    """
    code = SPLIT_CODE[split]
    base = D["A_use"] & (D["split"] == code)
    if split == "test":
        m = base & (D["A_stage"] <= t)
    elif upto:
        m = base & (D["A_stage"] >= 0) & (D["A_stage"] <= t)
    else:
        m = base & (D["A_stage"] == t)
    return D["F"][m], D["y"][m].astype(np.int64)


# ----------------------------------------------------------------------------- kịch bản B
def shared_ids_B(M):
    l2i = M["label2id"]
    return sorted(l2i[c] for c in M["shared_classes_B"] if c in l2i)


def select_B(D, k, split, remap):
    """Stage k (0 = ngày 1, 1 = ngày 2) của kịch bản B; nhãn được ánh xạ về 0..K-1 theo `remap`."""
    m = (D["B_stage"] == k) & (D["split"] == SPLIT_CODE[split])
    y = D["y"][m].astype(np.int64)
    keep = np.isin(y, list(remap.keys()))
    y = np.array([remap[v] for v in y[keep]], dtype=np.int64)
    return D["F"][m][keep], y


# ----------------------------------------------------------------------------- replay buffer
class ReplayBuffer:
    """Bộ nhớ cân bằng theo lớp: mỗi lớp giữ tối đa `per_class` mẫu (chọn ngẫu nhiên)."""

    def __init__(self, per_class=500, seed=0):
        self.k = per_class
        self.rng = np.random.default_rng(seed)
        self.X, self.y = None, None

    def add(self, X, y):
        Xs, ys = [], []
        for c in np.unique(y):
            idx = np.where(y == c)[0]
            pick = self.rng.choice(idx, min(self.k, len(idx)), replace=False)
            Xs.append(X[pick]); ys.append(y[pick])
        Xn, yn = np.concatenate(Xs), np.concatenate(ys)
        if self.X is not None:
            Xn, yn = np.concatenate([self.X, Xn]), np.concatenate([self.y, yn])
        # trần áp dụng cho TỔNG số mẫu mỗi lớp (Benign xuất hiện ở mọi stage nên sẽ cộng dồn nếu không cắt)
        keep = []
        for c in np.unique(yn):
            idx = np.where(yn == c)[0]
            keep.append(idx if len(idx) <= self.k else self.rng.choice(idx, self.k, replace=False))
        keep = np.sort(np.concatenate(keep))
        self.X, self.y = Xn[keep], yn[keep]

    def __len__(self):
        return 0 if self.X is None else len(self.y)


def balanced_weights(y, n_classes, power=1.0):
    """
    Trọng số lấy mẫu theo lớp = tần suất ** (-power).
      power=1.0: cân bằng hoàn toàn (mỗi lớp được lấy ngang nhau; buffer nhỏ bị lặp rất nhiều lần -> dễ quá khớp buffer)
      power=0.5: nghịch căn bậc hai (thoả hiệp, mặc định của cấu hình)
      power=0.0: lấy mẫu tự nhiên
    """
    cnt = np.bincount(y, minlength=n_classes).astype(np.float64)
    w = np.maximum(cnt, 1.0) ** (-power)
    return w[y]


# ----------------------------------------------------------------------------- metric
def f1_row(y_true, y_pred, groups):
    """Macro-F1 trên từng nhóm lớp (cột của ma trận R) + Macro-F1 toàn bộ lớp đã thấy."""
    row = [f1_score(y_true, y_pred, labels=g, average="macro", zero_division=0) for g in groups]
    all_ids = sorted({i for g in groups for i in g})
    overall = f1_score(y_true, y_pred, labels=all_ids, average="macro", zero_division=0)
    return row, overall


def per_class_recall(y_true, y_pred, ids):
    r = recall_score(y_true, y_pred, labels=ids, average=None, zero_division=0)
    return {int(i): float(v) for i, v in zip(ids, r)}


def il_metrics(R):
    """
    R[i][j] = Macro-F1 trên nhóm lớp j sau khi cập nhật xong stage i (chỉ dùng i >= j).
    Trả về: avg_f1 (trung bình hàng cuối), forgetting, BWT.
    """
    R = np.asarray(R, dtype=float)
    T = R.shape[0]
    avg_f1 = float(np.mean(R[T - 1, :T]))
    if T < 2:
        return dict(avg_f1=avg_f1, forgetting=0.0, bwt=0.0)
    forgetting = float(np.mean([np.max(R[j:T - 1, j]) - R[T - 1, j] for j in range(T - 1)]))
    bwt = float(np.mean([R[T - 1, j] - R[j, j] for j in range(T - 1)]))
    return dict(avg_f1=avg_f1, forgetting=forgetting, bwt=bwt)


# ----------------------------------------------------------------------------- batch learning theo split có sẵn
def select_view(D, M, view="A"):
    """
    Lấy train/val/test từ cột `split` do pipeline tiền xử lý tạo ra (chia theo KHỐI THỜI GIAN, không ngẫu nhiên).
      view="A"        : đúng tập dữ liệu của kịch bản A (lớp tấn công từ ngày 1; lớp chỉ có ngày 2 như Portmap lấy từ ngày 2)
                        -> kết quả = upper bound (joint retrain) của stage cuối
      view="all"      : mọi dòng của cả hai ngày; mỗi (ngày, lớp) có khối train/val/test riêng
      view="cross_day": các lớp dùng chung hai ngày; train/val = ngày 1, test = khối test của ngày 2 (đo drift thật)
    Trả về (data, ids, names): data[split] = (X, y) với y đã ánh xạ về 0..K-1; ids = id lớp gốc; names = tên lớp.
    """
    sp, day, y_all = D["split"], D["day"], D["y"].astype(np.int64)
    if view == "A":
        masks = {k: D["A_use"] & (sp == c) for k, c in SPLIT_CODE.items()}
    elif view == "all":
        masks = {k: (sp == c) for k, c in SPLIT_CODE.items()}
    elif view == "cross_day":
        sh = np.isin(y_all, shared_ids_B(M))
        masks = {"train": sh & (day == 1) & (sp == 0), "val": sh & (day == 1) & (sp == 1),
                 "test": sh & (day == 2) & (sp == 2)}
    else:
        raise ValueError(f"view={view!r}: chọn 'A', 'all' hoặc 'cross_day'")

    ids = sorted(set().union(*[set(np.unique(y_all[m]).tolist()) for m in masks.values()]))
    miss = sorted((set(np.unique(y_all[masks['val']]).tolist()) | set(np.unique(y_all[masks['test']]).tolist()))
                  - set(np.unique(y_all[masks['train']]).tolist()))
    if miss:
        names_all = M["classes"]
        raise ValueError(f"Lớp có ở val/test nhưng KHÔNG có ở train: {[names_all[i] for i in miss]}")
    lut = np.full(y_all.max() + 1, -1, dtype=np.int64)
    for i, g in enumerate(ids):
        lut[g] = i
    data = {k: (D["F"][m], lut[y_all[m]]) for k, m in masks.items()}
    return data, ids, [M["classes"][g] for g in ids]


def split_table(data, names):
    """Bảng số mẫu: lớp x (train, val, test)."""
    import pandas as pd
    K = len(names)
    t = pd.DataFrame({sp: np.bincount(y, minlength=K) for sp, (_, y) in data.items()}, index=names)
    t["total"] = t.sum(axis=1)
    return t


def wilson_ci(k, n, z=1.96):
    """Khoảng tin cậy Wilson 95% cho tỉ lệ k/n (quan trọng với lớp ít mẫu như Portmap)."""
    if n == 0:
        return 0.0, 1.0
    p = k / n
    den = 1 + z * z / n
    centre = p + z * z / (2 * n)
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return float((centre - half) / den), float((centre + half) / den)


def evaluate_predictions(y_true, y_pred, names):
    """Báo cáo đầy đủ: Accuracy, Macro/Weighted-F1, bảng theo lớp (kèm CI của recall), confusion matrix."""
    import pandas as pd
    from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                                 precision_recall_fscore_support)
    labels = list(range(len(names)))
    p, r, f, n = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    rows = []
    for i, nm in enumerate(names):
        k = int(round(r[i] * n[i]))
        lo, hi = wilson_ci(k, int(n[i]))
        rows.append(dict(lop=nm, n_test=int(n[i]), precision=p[i], recall=r[i], f1=f[i],
                         recall_ci_low=lo, recall_ci_high=hi))
    return dict(
        accuracy=float(accuracy_score(y_true, y_pred)),
        macro_f1=float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        weighted_f1=float(f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)),
        per_class=pd.DataFrame(rows),
        confusion=pd.DataFrame(confusion_matrix(y_true, y_pred, labels=labels), index=names, columns=names),
        report=classification_report(y_true, y_pred, labels=labels, target_names=names, zero_division=0, digits=4),
    )


def export_misclassified(X, y_true, y_pred, names, path):
    """Giống export_misclassified_to_txt của Trainer trong start.py, nhưng nhận sẵn dự đoán."""
    bad = np.where(y_true != y_pred)[0]
    with open(path, "w", encoding="utf-8") as f:
        f.write("True_Label\tPredicted_Label\tFeatures_Tokenized\n")
        for i in bad:
            f.write(f"{names[y_true[i]]}\t{names[y_pred[i]]}\t{','.join(map(str, X[i].tolist()))}\n")
    return len(bad)

## LÕI INCREMENTAL LEARNING (bọc quanh Deep_MHA_Tabular, không đổi kiến trúc)


In [ ]:
import copy
import json
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.metrics import f1_score

# start.py / il_utils đã được định nghĩa ở các cell phía trên (cùng namespace)
S = types.SimpleNamespace(FocalLoss=FocalLoss, Trainer=Trainer,
                          MonotonicTabularEmbedding=MonotonicTabularEmbedding,
                          Deep_MHA_Tabular=Deep_MHA_Tabular)

IL_METHODS = ["naive", "replay", "lwf", "replay_kd", "replay_kd_feat"]


# =============================================================================
# 0. CẤU HÌNH (khớp config trong start.py; thêm phần IL)
# =============================================================================
def build_cfg(M, **overrides):
    tk = M["tokenizer"]
    cfg = dict(
        # --- kiến trúc (giống start.py) ---
        n_features=len(M["features"]), num_bins=tk["num_bins"],
        value_dim=192, feat_dim=64, n_segments=192,
        num_layers=4, num_branches=4, num_heads_per_branch=4,
        mlp_dropout=0.15, value_embed="ple",
        per_feature_norm=False,         # False = ĐÚNG kiến trúc start.py (mặc định). True = tuỳ chọn thử nghiệm: token/feat_max theo từng cột trước PLE
        # --- huấn luyện batch (stage 0 và joint retrain) ---
        batch_size=256, epochs=50, patience=12, max_lr=1e-3, weight_decay=1e-2,        # khớp config trong start.py
        label_smoothing=0.05, focal_gamma=1.5, grad_clip=1.0,
        balanced_sampler=True, balance_power=0.5,   # 0=tự nhiên, 1=cân bằng hoàn toàn
        # --- cập nhật tăng dần ---
        il_epochs=5, il_lr=1e-4, kd_weight=1.0, kd_T=2.0, feat_weight=1.0,
        buffer_per_class=500, freeze=None,     # freeze: None | "head" | "last"
    )
    cfg.update(overrides)
    return cfg


def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =============================================================================
# 1. MÔ HÌNH
# =============================================================================
class PerFeatureEmbedding(S.MonotonicTabularEmbedding):
    """
    Giống MonotonicTabularEmbedding nhưng t = token / feat_max[cột] (clamp 0..1) thay vì token / (NUM_BINS-1).
    Lý do: cột nhị phân/low_card/port chỉ có token 0..1 / 0..19 / 0..6; chia cho NUM_BINS-1 = 30000 làm
    t ~ 3e-5 nên các giá trị của cột rời rạc gần như trùng nhau trong không gian embedding.
    feat_max do notebook tiền xử lý tính (meta.json -> tokenizer.feat_max), chỉ suy ra từ tập fit.
    """
    def __init__(self, feat_max, **kw):
        super().__init__(**kw)
        fm = torch.as_tensor(np.maximum(np.asarray(feat_max, dtype=np.float32), 1.0))
        self.register_buffer("feat_max", fm)

    def forward(self, x_tokens):
        t = (x_tokens.float() / self.feat_max).clamp(0.0, 1.0)                 # (B, F)
        ve = self.value_embedding
        if hasattr(ve, "_ple_encode"):                                         # PLEValueEmbedding
            val = ve.proj(ve._ple_encode(t))
        else:                                                                  # SimpleValueEmbedding
            val = ve.net(t.unsqueeze(-1))
        idx = torch.arange(self.num_features, device=x_tokens.device)
        feat = self.feature_embedding(idx).unsqueeze(0).expand(x_tokens.size(0), -1, -1)
        return torch.cat([val, feat], dim=-1)


class IncrementalDeepMHA(S.Deep_MHA_Tabular):
    """Deep_MHA_Tabular + trích biểu diễn (cho distillation) + mở rộng đầu phân loại."""
    def __init__(self, feat_max=None, **kw):
        super().__init__(**kw)
        if feat_max is not None:
            self.feature_embedder = PerFeatureEmbedding(
                feat_max,
                num_features=kw["num_numeric_features"], num_bins=kw["num_bins"],
                value_dim=kw["value_dim"], feat_dim=kw["feat_dim"],
                n_segments=kw.get("n_segments"), value_embed=kw.get("value_embed", "ple"))

    def forward_features(self, tokenized_x):
        """Giống forward() của Deep_MHA_Tabular, dừng trước classifier: trả về vector pooled (3 * embed_dim)."""
        x = self.feature_embedder(tokenized_x)
        x = self.input_norm(x)
        for layer in self.layers:
            x = layer(x)
        x = self.final_norm(x)
        attn_w = torch.softmax(self.attn_pool_w(x), dim=1)
        attn_pooled = (x * attn_w).sum(dim=1)
        max_pooled, _ = x.max(dim=1)
        mean_pooled = x.mean(dim=1)
        return torch.cat([attn_pooled, max_pooled, mean_pooled], dim=1)

    def forward(self, tokenized_x):
        return self.classifier(self.forward_features(tokenized_x))

    def expand_head(self, n_new):
        """Thêm n_new neuron đầu ra, GIỮ NGUYÊN trọng số của các lớp cũ."""
        old = self.classifier[-1]
        new = nn.Linear(old.in_features, old.out_features + n_new).to(old.weight.device)
        with torch.no_grad():
            new.weight[: old.out_features] = old.weight
            new.bias[: old.out_features] = old.bias
        self.classifier[-1] = new


def new_model(cfg, n_classes, feat_max, device):
    model = IncrementalDeepMHA(
        feat_max=(feat_max if cfg["per_feature_norm"] else None),
        num_numeric_features=cfg["n_features"], num_bins=cfg["num_bins"],
        value_dim=cfg["value_dim"], feat_dim=cfg["feat_dim"], n_segments=cfg["n_segments"],
        num_layers=cfg["num_layers"], num_branches=cfg["num_branches"],
        num_heads_per_branch=cfg["num_heads_per_branch"], num_classes=n_classes,
        mlp_dropout=cfg["mlp_dropout"], value_embed=cfg["value_embed"]).to(device)
    n_par = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   -> mô hình: {n_classes} lớp | {n_par:,} tham số | per_feature_norm={cfg['per_feature_norm']}")
    return model


# =============================================================================
# 2. DỮ LIỆU / DỰ ĐOÁN
# =============================================================================
def make_loader(X, y, batch_size, shuffle=False, weights=None):
    ds = TensorDataset(torch.from_numpy(np.ascontiguousarray(X, dtype=np.int64)),
                       torch.from_numpy(np.ascontiguousarray(y, dtype=np.int64)))
    pin = torch.cuda.is_available()
    if weights is not None:
        sampler = WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double),
                                        num_samples=len(ds), replacement=True)
        return DataLoader(ds, batch_size=batch_size, sampler=sampler, pin_memory=pin, num_workers=0)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, pin_memory=pin, num_workers=0)


@torch.no_grad()
def predict(model, X, device, bs=2048):
    model.eval()
    out = []
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(np.ascontiguousarray(X[i:i + bs], dtype=np.int64)).to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
            out.append(model(xb).argmax(1).cpu())
    return torch.cat(out).numpy()


def _sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize()


# =============================================================================
# 3. HUẤN LUYỆN BATCH (stage 0 và joint retrain) — dùng Trainer/FocalLoss/OneCycleLR của bạn
# =============================================================================
def train_batch(model, Xtr, ytr, Xva, yva, cfg, device):
    n_cls = model.classifier[-1].out_features
    w = balanced_weights(ytr, n_cls, cfg["balance_power"]) if cfg["balanced_sampler"] else None
    tl = make_loader(Xtr, ytr, cfg["batch_size"], shuffle=True, weights=w)
    vl = make_loader(Xva, yva, cfg["batch_size"], shuffle=False)
    crit = S.FocalLoss(gamma=cfg["focal_gamma"], label_smoothing=cfg["label_smoothing"])
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["max_lr"], weight_decay=cfg["weight_decay"])
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=cfg["max_lr"], epochs=cfg["epochs"], steps_per_epoch=len(tl),
        pct_start=0.20, anneal_strategy="cos", div_factor=20.0, final_div_factor=250.0)
    trainer = S.Trainer(model, crit, opt, sched, device, scheduler_per_batch=True, grad_clip=cfg["grad_clip"])
    trainer.fit(tl, vl, epochs=cfg["epochs"], patience=cfg["patience"])
    return model


# =============================================================================
# 4. CẬP NHẬT TĂNG DẦN
# =============================================================================
def kd_loss(student_logits, teacher_logits, T):
    return Fnn.kl_div(Fnn.log_softmax(student_logits.float() / T, dim=1),
                      Fnn.softmax(teacher_logits.float() / T, dim=1),
                      reduction="batchmean") * (T * T)


def set_trainable(model, mode):
    """mode: None (tất cả) | 'head' (chỉ classifier) | 'last' (tầng Transformer cuối + norm + pooling + classifier)."""
    last = f"layers.{len(model.layers) - 1}."
    for n, p in model.named_parameters():
        if mode is None:
            p.requires_grad_(True)
        elif mode == "head":
            p.requires_grad_(n.startswith("classifier"))
        elif mode == "last":
            p.requires_grad_(n.startswith(("classifier", "final_norm", "attn_pool_w", last)))
        else:
            raise ValueError(f"freeze={mode!r} không hợp lệ")


def il_update(model, Xn, yn, buf, n_old, n_total, method, cfg, device):
    """
    Cập nhật `model` bằng dữ liệu stage mới (Xn, yn). method:
      naive          : fine-tune thuần (lower bound)
      replay         : trộn thêm buffer cân bằng lớp
      lwf            : distillation logits lớp cũ, KHÔNG dùng buffer
      replay_kd      : replay + distillation logits lớp cũ
      replay_kd_feat : replay + distillation logits + distillation vector pooled của MHA
    Trả về (model, thời gian cập nhật tính bằng giây).
    """
    assert method in IL_METHODS, method
    use_replay = method in ("replay", "replay_kd", "replay_kd_feat")
    use_kd = method in ("lwf", "replay_kd", "replay_kd_feat")
    use_feat = method == "replay_kd_feat"
    cuda = device.type == "cuda"

    _sync(device); t0 = time.time()
    teacher = None
    if use_kd:
        teacher = copy.deepcopy(model).eval()                  # đóng băng bản TRƯỚC khi mở rộng đầu
        for p in teacher.parameters():
            p.requires_grad_(False)
    if n_total > n_old:
        model.expand_head(n_total - n_old)

    if use_replay and len(buf) > 0:
        X, y = np.concatenate([Xn, buf.X]), np.concatenate([yn, buf.y])
    else:
        X, y = Xn, yn
    loader = make_loader(X, y, cfg["batch_size"], weights=balanced_weights(y, n_total, cfg["balance_power"]))

    set_trainable(model, cfg["freeze"])
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["il_lr"], weight_decay=cfg["weight_decay"])
    crit = S.FocalLoss(gamma=cfg["focal_gamma"], label_smoothing=cfg["label_smoothing"])
    scaler = torch.amp.GradScaler("cuda") if cuda else None

    model.train()
    for _ in range(cfg["il_epochs"]):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=cuda):
                feats = model.forward_features(xb)
                logits = model.classifier(feats)
                loss = crit(logits, yb)
                if teacher is not None:
                    with torch.no_grad():
                        t_feats = teacher.forward_features(xb)
                        t_logits = teacher.classifier(t_feats)
                    loss = loss + cfg["kd_weight"] * kd_loss(logits[:, :n_old], t_logits, cfg["kd_T"])
                    if use_feat:
                        loss = loss + cfg["feat_weight"] * Fnn.mse_loss(feats.float(), t_feats.float())
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, max_norm=cfg["grad_clip"])
                scaler.step(opt); scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(params, max_norm=cfg["grad_clip"])
                opt.step()

    set_trainable(model, None)
    if use_replay:
        buf.add(Xn, yn)                                        # cập nhật buffer SAU khi train
    _sync(device)
    return model, time.time() - t0


# =============================================================================
# 5. KỊCH BẢN A: CLASS-INCREMENTAL
# =============================================================================
def _nan_to_none(R):
    return [[None if np.isnan(x) else float(x) for x in row] for row in R]


def run_class_incremental(D, M, cfg, methods, seed, device, out_dir, run_joint=True):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    stages, benign = class_ids_by_stage(M)
    T = len(stages)
    assert T >= 2, "Cần ít nhất 2 stage có dữ liệu."
    groups = [sorted({benign, *stages[0]})] + [sorted(s) for s in stages[1:]]
    feat_max = np.asarray(M["tokenizer"]["feat_max"], dtype=np.float32)
    n_seen = [len(seen_ids(stages, benign, t)) for t in range(T)]

    print(f"\n{'=' * 78}\nKỊCH BẢN A | seed={seed} | {T} stage | số lớp tích lũy: {n_seen}\n{'=' * 78}")
    seed_everything(seed)
    X0, y0 = select_A(D, 0, "train"); Xv0, yv0 = select_A(D, 0, "val")
    base = new_model(cfg, n_seen[0], feat_max, device)
    _sync(device); t0 = time.time()
    train_batch(base, X0, y0, Xv0, yv0, cfg, device)
    _sync(device); t_stage0 = time.time() - t0

    Xe0, ye0 = select_A(D, 0, "test")
    row0, _ = f1_row(ye0, predict(base, Xe0, device), groups[:1])

    details, rows = {}, []
    for method in methods:
        seed_everything(seed + 1)
        model = copy.deepcopy(base)
        buf = ReplayBuffer(cfg["buffer_per_class"], seed)
        buf.add(X0, y0)
        R = np.full((T, T), np.nan); R[0, 0] = row0[0]
        upd_t, overall = [], []
        for t in range(1, T):
            n_old, n_tot = n_seen[t - 1], n_seen[t]
            Xn, yn = select_A(D, t, "train")
            model, dt = il_update(model, Xn, yn, buf, n_old, n_tot, method, cfg, device)
            Xe, ye = select_A(D, t, "test")
            pred = predict(model, Xe, device)
            r, ov = f1_row(ye, pred, groups[: t + 1])
            R[t, : t + 1] = r
            upd_t.append(dt); overall.append(ov)
            print(f"   [{method:<15}] stage {t}: update {dt:6.1f}s | Macro-F1 toàn bộ = {ov:.4f} | "
                  f"từng nhóm = {[round(v, 3) for v in r]}")
        m = il_metrics(R)
        rec = per_class_recall(ye, pred, seen_ids(stages, benign, T - 1))
        details[method] = dict(R=_nan_to_none(R), update_seconds=upd_t, overall_f1_per_stage=overall,
                               final_recall_per_class=rec, buffer_size=len(buf), **m)
        rows.append(dict(scenario="A", seed=seed, method=method, avg_macro_f1=m["avg_f1"],
                         forgetting=m["forgetting"], bwt=m["bwt"], final_overall_f1=overall[-1],
                         total_update_s=float(np.sum(upd_t)), buffer_size=len(buf)))

    if run_joint:                                           # upper bound = Batch Learning: huấn luyện lại từ đầu
        R = np.full((T, T), np.nan); R[0, 0] = row0[0]
        times, overall = [t_stage0], []
        for t in range(1, T):
            seed_everything(seed + 2)
            Xj, yj = select_A(D, t, "train", upto=True); Xjv, yjv = select_A(D, t, "val", upto=True)
            jm = new_model(cfg, n_seen[t], feat_max, device)
            _sync(device); t0 = time.time()
            train_batch(jm, Xj, yj, Xjv, yjv, cfg, device)
            _sync(device); times.append(time.time() - t0)
            Xe, ye = select_A(D, t, "test")
            pred = predict(jm, Xe, device)
            r, ov = f1_row(ye, pred, groups[: t + 1])
            R[t, : t + 1] = r; overall.append(ov)
            print(f"   [joint_retrain  ] stage {t}: train {times[-1]:6.1f}s | Macro-F1 toàn bộ = {ov:.4f}")
            del jm
        m = il_metrics(R)
        details["joint_retrain"] = dict(R=_nan_to_none(R), update_seconds=times[1:], overall_f1_per_stage=overall, **m)
        rows.append(dict(scenario="A", seed=seed, method="joint_retrain", avg_macro_f1=m["avg_f1"],
                         forgetting=m["forgetting"], bwt=m["bwt"], final_overall_f1=overall[-1],
                         total_update_s=float(np.sum(times[1:])), buffer_size=0))

    (out_dir / f"A_seed{seed}.json").write_text(json.dumps(
        dict(config=cfg, stage0_train_seconds=t_stage0, groups=groups, details=details), indent=2, default=str))
    return rows, details


# =============================================================================
# 6. KỊCH BẢN B: DRIFT THEO NGÀY (không thêm lớp mới)
# =============================================================================
def run_domain_drift(D, M, cfg, methods, seed, device, out_dir, run_joint=True):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    shared = shared_ids_B(M)
    remap = {g: i for i, g in enumerate(shared)}
    K = len(shared)
    names = {v: k for k, v in M["label2id"].items()}
    print(f"\n{'=' * 78}\nKỊCH BẢN B | seed={seed} | lớp dùng chung: {[names[g] for g in shared]}\n{'=' * 78}")
    feat_max = np.asarray(M["tokenizer"]["feat_max"], dtype=np.float32)

    X0, y0 = select_B(D, 0, "train", remap); Xv0, yv0 = select_B(D, 0, "val", remap)
    X1, y1 = select_B(D, 1, "train", remap); Xv1, yv1 = select_B(D, 1, "val", remap)
    Xt0, yt0 = select_B(D, 0, "test", remap); Xt1, yt1 = select_B(D, 1, "test", remap)
    labels = list(range(K))

    def evaluate(model):
        p0, p1 = predict(model, Xt0, device), predict(model, Xt1, device)
        return dict(f1_day1=float(f1_score(yt0, p0, labels=labels, average="macro", zero_division=0)),
                    f1_day2=float(f1_score(yt1, p1, labels=labels, average="macro", zero_division=0)),
                    recall_day2={names[shared[i]]: v for i, v in per_class_recall(yt1, p1, labels).items()})

    seed_everything(seed)
    base = new_model(cfg, K, feat_max, device)
    train_batch(base, X0, y0, Xv0, yv0, cfg, device)

    res = {"no_update": dict(**evaluate(base), update_seconds=0.0)}
    for method in methods:
        seed_everything(seed + 1)
        model = copy.deepcopy(base)
        buf = ReplayBuffer(cfg["buffer_per_class"], seed); buf.add(X0, y0)
        model, dt = il_update(model, X1, y1, buf, K, K, method, cfg, device)
        res[method] = dict(**evaluate(model), update_seconds=dt)
    if run_joint:
        seed_everything(seed + 2)
        jm = new_model(cfg, K, feat_max, device)
        _sync(device); t0 = time.time()
        train_batch(jm, np.concatenate([X0, X1]), np.concatenate([y0, y1]),
                    np.concatenate([Xv0, Xv1]), np.concatenate([yv0, yv1]), cfg, device)
        _sync(device)
        res["joint_retrain"] = dict(**evaluate(jm), update_seconds=time.time() - t0)

    rows = [dict(scenario="B", seed=seed, method=k, f1_day1=v["f1_day1"], f1_day2=v["f1_day2"],
                 update_s=v["update_seconds"]) for k, v in res.items()]
    for r in rows:
        print(f"   [{r['method']:<15}] Macro-F1 ngày 1 = {r['f1_day1']:.4f} | ngày 2 = {r['f1_day2']:.4f} | "
              f"{r['update_s']:.1f}s")
    (out_dir / f"B_seed{seed}.json").write_text(json.dumps(dict(config=cfg, results=res), indent=2, default=str))
    return rows, res

## XUẤT KẾT QUẢ CHI TIẾT KỊCH BẢN A (ma trận nhầm lẫn + thông số theo từng stage)
Dán cell này NGAY SAU cell "LÕI INCREMENTAL LEARNING" (cell định nghĩa run_class_incremental).
Cell định nghĩa lại run_class_incremental: thuật toán, thứ tự seed và kết quả số GIỮ NGUYÊN, chỉ thêm phần xuất file.

Với mỗi seed, mỗi phương pháp (kể cả joint_retrain) và mỗi stage, ghi vào RESULTS/il_A/seed{seed}/{method}/:
  stage{t}_confusion.csv        ma trận nhầm lẫn (số lượng), hàng = nhãn thật, cột = dự đoán
  stage{t}_confusion_norm.png   heatmap chuẩn hóa theo hàng (recall), kèm số lượng
  stage{t}_per_class.csv        precision / recall / F1 / support / CI 95% của recall cho từng lớp
  stage{t}_misclassified.csv    tối đa 2000 mẫu sai (tuỳ chọn, EXPORT_MISCLASSIFIED=True)
  R_matrix.csv                  ma trận R (Macro-F1 nhóm lớp j sau stage i)
Và các bảng tổng hợp ở RESULTS/:
  il_A_stage_metrics_seed{seed}.csv     accuracy, macro/weighted F1, macro precision/recall, thời gian - theo stage
  il_A_final_per_class_f1_seed{seed}.csv F1 từng lớp ở stage cuối, mỗi cột một phương pháp


In [ ]:
import copy, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score

EXPORT_PNG = True             # vẽ heatmap ma trận nhầm lẫn
EXPORT_MISCLASSIFIED = False  # xuất mẫu bị phân loại sai (file lớn hơn)


def _plot_confusion(cm_df, path, title):
    cm = cm_df.to_numpy().astype(float)
    norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    n = len(cm_df)
    fig, ax = plt.subplots(figsize=(0.62 * n + 3.2, 0.55 * n + 2.6))
    im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(n)); ax.set_xticklabels(cm_df.columns, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(n)); ax.set_yticklabels(cm_df.index, fontsize=9)
    ax.set_xlabel("Dự đoán"); ax.set_ylabel("Nhãn thật"); ax.set_title(title, fontsize=10)
    for i in range(n):
        for j in range(n):
            if cm[i, j] > 0:
                ax.text(j, i, f"{norm[i, j]:.2f}\n({int(cm[i, j])})", ha="center", va="center", fontsize=6.5,
                        color="white" if norm[i, j] > 0.5 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Tỉ lệ theo hàng (recall)")
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)


def export_stage_eval(y_true, y_pred, names, out_dir, t, title, X=None):
    """Đánh giá đầy đủ một stage, ghi file, trả về dict thông số tổng hợp."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    ev = evaluate_predictions(y_true, y_pred, names)
    ev["confusion"].to_csv(out_dir / f"stage{t}_confusion.csv")
    ev["per_class"].to_csv(out_dir / f"stage{t}_per_class.csv", index=False)
    if EXPORT_PNG:
        _plot_confusion(ev["confusion"], out_dir / f"stage{t}_confusion_norm.png", title)
    if EXPORT_MISCLASSIFIED and X is not None:
        export_misclassified(X, y_true, y_pred, names, out_dir / f"stage{t}_misclassified.csv")
    labels = list(range(len(names)))
    return dict(n_classes=len(names), n_test=int(len(y_true)), accuracy=ev["accuracy"], macro_f1=ev["macro_f1"],
                weighted_f1=ev["weighted_f1"],
                macro_precision=float(precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
                macro_recall=float(recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
                per_class_f1=dict(zip(ev["per_class"]["lop"], ev["per_class"]["f1"].round(4)))), ev


def run_class_incremental(D, M, cfg, methods, seed, device, out_dir, run_joint=True):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    exp_dir = out_dir / "il_A" / f"seed{seed}"
    stages, benign = class_ids_by_stage(M)
    T = len(stages)
    assert T >= 2, "Cần ít nhất 2 stage có dữ liệu."
    groups = [sorted({benign, *stages[0]})] + [sorted(s) for s in stages[1:]]
    feat_max = np.asarray(M["tokenizer"]["feat_max"], dtype=np.float32)
    n_seen = [len(seen_ids(stages, benign, t)) for t in range(T)]
    for t in range(T):                                   # id lớp đã thấy luôn là 0..n_seen-1 (label2id theo thứ tự stage)
        assert seen_ids(stages, benign, t) == list(range(n_seen[t])), "id lớp không liên tục theo stage"
    stage_names = [[M["classes"][i] for i in range(n_seen[t])] for t in range(T)]

    print(f"\n{'=' * 78}\nKỊCH BẢN A | seed={seed} | {T} stage | số lớp tích lũy: {n_seen}\n{'=' * 78}")
    seed_everything(seed)
    X0, y0 = select_A(D, 0, "train"); Xv0, yv0 = select_A(D, 0, "val")
    base = new_model(cfg, n_seen[0], feat_max, device)
    _sync(device); t0 = time.time()
    train_batch(base, X0, y0, Xv0, yv0, cfg, device)
    _sync(device); t_stage0 = time.time() - t0

    Xe0, ye0 = select_A(D, 0, "test")
    pred0 = predict(base, Xe0, device)
    row0, _ = f1_row(ye0, pred0, groups[:1])
    s0, _ = export_stage_eval(ye0, pred0, stage_names[0], exp_dir / "stage0_base", 0,
                              f"seed {seed} | stage 0 (mô hình nền)", Xe0)
    stage_rows = [dict(seed=seed, method="stage0_base", stage=0, update_s=t_stage0,
                       **{k: v for k, v in s0.items() if k != "per_class_f1"})]
    final_pc = {}

    def _run_chain(label, step_fn):
        """step_fn(t) -> (pred, ye, Xe, seconds). Ghi file mọi stage, trả về R, thời gian, overall."""
        R = np.full((T, T), np.nan); R[0, 0] = row0[0]
        upd_t, overall = [], []
        for t in range(1, T):
            pred, ye, Xe, dt = step_fn(t)
            r, ov = f1_row(ye, pred, groups[: t + 1])
            R[t, : t + 1] = r
            upd_t.append(dt); overall.append(ov)
            st, ev = export_stage_eval(ye, pred, stage_names[t], exp_dir / label, t,
                                       f"seed {seed} | {label} | stage {t} ({n_seen[t]} lớp)", Xe)
            stage_rows.append(dict(seed=seed, method=label, stage=t, update_s=dt,
                                   **{k: v for k, v in st.items() if k != "per_class_f1"}))
            if t == T - 1:
                final_pc[label] = st["per_class_f1"]
            print(f"   [{label:<15}] stage {t}: {dt:6.1f}s | acc = {st['accuracy']:.4f} | "
                  f"Macro-F1 = {ov:.4f} | W-F1 = {st['weighted_f1']:.4f} | nhóm = {[round(v, 3) for v in r]}")
        pd.DataFrame(R, index=[f"sau_stage{i}" for i in range(T)],
                     columns=[f"nhom{j}" for j in range(T)]).to_csv(exp_dir / label / "R_matrix.csv")
        return R, upd_t, overall

    details, rows = {}, []
    for method in methods:
        seed_everything(seed + 1)
        model = copy.deepcopy(base)
        buf = ReplayBuffer(cfg["buffer_per_class"], seed)
        buf.add(X0, y0)
        state = {}

        def step(t, model_ref=[model]):
            n_old, n_tot = n_seen[t - 1], n_seen[t]
            Xn, yn = select_A(D, t, "train")
            model_ref[0], dt = il_update(model_ref[0], Xn, yn, buf, n_old, n_tot, method, cfg, device)
            Xe, ye = select_A(D, t, "test")
            pred = predict(model_ref[0], Xe, device)
            state.update(ye=ye, pred=pred)
            return pred, ye, Xe, dt

        R, upd_t, overall = _run_chain(method, step)
        m = il_metrics(R)
        rec = per_class_recall(state["ye"], state["pred"], seen_ids(stages, benign, T - 1))
        details[method] = dict(R=_nan_to_none(R), update_seconds=upd_t, overall_f1_per_stage=overall,
                               final_recall_per_class=rec, buffer_size=len(buf), **m)
        rows.append(dict(scenario="A", seed=seed, method=method, avg_macro_f1=m["avg_f1"],
                         forgetting=m["forgetting"], bwt=m["bwt"], final_overall_f1=overall[-1],
                         total_update_s=float(np.sum(upd_t)), buffer_size=len(buf)))

    if run_joint:                                           # upper bound = Batch Learning: huấn luyện lại từ đầu
        def jstep(t):
            seed_everything(seed + 2)
            Xj, yj = select_A(D, t, "train", upto=True); Xjv, yjv = select_A(D, t, "val", upto=True)
            jm = new_model(cfg, n_seen[t], feat_max, device)
            _sync(device); t0 = time.time()
            train_batch(jm, Xj, yj, Xjv, yjv, cfg, device)
            _sync(device); dt = time.time() - t0
            Xe, ye = select_A(D, t, "test")
            pred = predict(jm, Xe, device)
            del jm
            return pred, ye, Xe, dt

        R, times, overall = _run_chain("joint_retrain", jstep)
        m = il_metrics(R)
        details["joint_retrain"] = dict(R=_nan_to_none(R), update_seconds=times, overall_f1_per_stage=overall, **m)
        rows.append(dict(scenario="A", seed=seed, method="joint_retrain", avg_macro_f1=m["avg_f1"],
                         forgetting=m["forgetting"], bwt=m["bwt"], final_overall_f1=overall[-1],
                         total_update_s=float(np.sum(times)), buffer_size=0))

    # ---- bảng tổng hợp của seed này
    st_df = pd.DataFrame(stage_rows)
    st_df.to_csv(out_dir / f"il_A_stage_metrics_seed{seed}.csv", index=False)
    pc_df = pd.DataFrame(final_pc)
    pc_df.index.name = "lop"
    pc_df.to_csv(out_dir / f"il_A_final_per_class_f1_seed{seed}.csv")
    print(f"\nThông số theo stage (seed {seed}):")
    print(st_df.pivot_table(index="method", columns="stage", values="macro_f1").round(4).to_string())
    print(f"\nF1 từng lớp ở stage cuối (seed {seed}):")
    print(pc_df.round(3).to_string())
    print(f"\nĐã ghi ma trận nhầm lẫn + bảng theo lớp vào {exp_dir}")

    (out_dir / f"A_seed{seed}.json").write_text(json.dumps(
        dict(config=cfg, stage0_train_seconds=t_stage0, groups=groups, details=details,
             stage_metrics=stage_rows), indent=2, default=str))
    return rows, details


print("Đã định nghĩa lại run_class_incremental (có xuất ma trận nhầm lẫn + thông số theo stage)")


## NẠP DỮ LIỆU ĐÃ XỬ LÝ


In [ ]:
import numpy as np
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS = WORK_DIR / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

D, M = load_processed(PROCESSED)
cfg = build_cfg(M, **CFG_OVERRIDES)
feat_max = np.asarray(M["tokenizer"]["feat_max"], dtype=np.float32)
print(f"Thiết bị: {device} | token: {D['F'].shape} | NUM_BINS = {cfg['num_bins']:,} | lớp: {M['label2id']}")
print("Cấu hình:", json.dumps(cfg, default=str))

stages, benign = class_ids_by_stage(M)
print("\nKích thước từng stage (kịch bản A, split theo khối thời gian):")
for t in range(len(stages)):
    n_tr = len(select_A(D, t, "train")[1]); n_va = len(select_A(D, t, "val")[1]); n_te = len(select_A(D, t, "test")[1])
    print(f"  stage {t}: lớp mới {[M['classes'][i] for i in stages[t]]} | train {n_tr:,} | val {n_va:,} | test tích lũy {n_te:,}")


## BATCH LEARNING trên split có sẵn (thay phần __main__ của start.py)


In [ ]:
batch_rows = []
if RUN_BATCH:
    for view in BATCH_VIEWS:
        data, ids, names = select_view(D, M, view)
        (Xtr, ytr), (Xva, yva), (Xte, yte) = data["train"], data["val"], data["test"]
        tab = split_table(data, names)
        tab.to_csv(RESULTS / f"split_counts_view{view}.csv")
        print(f"\n{'#' * 80}\nVIEW = {view} | train={len(ytr):,} val={len(yva):,} test={len(yte):,} "
              f"(chia theo KHỐI THỜI GIAN, không ngẫu nhiên)\n{tab.to_string()}")
        small = tab[tab["test"] < 200]
        if len(small):
            print("CẢNH BÁO: lớp ít mẫu test (xem CI của recall):", {n: int(r["test"]) for n, r in small.iterrows()})

        for seed in SEEDS:
            print(f"\n{'=' * 80}\nVIEW {view} | SEED {seed}\n{'=' * 80}")
            seed_everything(seed)
            model = new_model(cfg, len(names), feat_max, device)
            t0 = time.time()
            train_batch(model, Xtr, ytr, Xva, yva, cfg, device)       # Trainer + FocalLoss + OneCycleLR của start.py
            if device.type == "cuda": torch.cuda.synchronize()
            train_s = time.time() - t0

            res = {}
            for split_name, (X_, y_) in (("val", (Xva, yva)), ("test", (Xte, yte))):
                t1 = time.time()
                pred = predict(model, X_, device)
                if device.type == "cuda": torch.cuda.synchronize()
                infer_s = time.time() - t1
                r = evaluate_predictions(y_, pred, names); res[split_name] = r
                print(f"\n--- {split_name.upper()} ---\n{r['report']}\nConfusion Matrix:\n{r['confusion'].to_string()}")
                print(f"\nAccuracy: {r['accuracy']:.4f} | Weighted F1: {r['weighted_f1']:.4f} | Macro F1: {r['macro_f1']:.4f}")
                r["per_class"].to_csv(RESULTS / f"per_class_{split_name}_view{view}_seed{seed}.csv", index=False)
                r["confusion"].to_csv(RESULTS / f"confusion_{split_name}_view{view}_seed{seed}.csv")
                if split_name == "test":
                    test_pred, test_infer_s = pred, infer_s
            mis = RESULTS / f"misclassified_test_view{view}_seed{seed}.txt"
            print(f"Đã ghi {export_misclassified(Xte, yte, test_pred, names, mis):,} mẫu dự đoán sai -> {mis.name}")
            torch.save(dict(state_dict=model.state_dict(), cfg=cfg, classes=names, class_ids=ids,
                            feat_max=feat_max.tolist(), num_bins=cfg["num_bins"], view=view),
                       RESULTS / f"best_mha_view{view}_seed{seed}.pt")
            batch_rows.append(dict(view=view, seed=seed, n_train=len(ytr), n_val=len(yva), n_test=len(yte),
                                   val_macro_f1=res["val"]["macro_f1"], test_acc=res["test"]["accuracy"],
                                   test_macro_f1=res["test"]["macro_f1"], test_weighted_f1=res["test"]["weighted_f1"],
                                   train_seconds=train_s, infer_ms_per_1k=1e6 * test_infer_s / max(len(yte), 1)))
            del model
            if device.type == "cuda": torch.cuda.empty_cache()
    df_batch = pd.DataFrame(batch_rows)
    df_batch.to_csv(RESULTS / "summary_batch.csv", index=False)
    print("\n=== TỔNG HỢP BATCH ===\n" + df_batch.round(4).to_string(index=False))
else:
    print("Bỏ qua Batch Learning (RUN_BATCH=False)")

## BASELINE RANDOM FOREST + ENSEMBLE VỚI MHA
Cùng token, cùng split (select_view). Trọng số ensemble chọn trên VAL rồi mới đo test.

In [ ]:
# ======================================================================================
# BASELINE RANDOM FOREST + ENSEMBLE VỚI MHA
# Dán thành cell SAU cell "BATCH LEARNING", trước "ĐÓNG GÓI KẾT QUẢ".
# Dùng đúng select_view / new_model / checkpoint của notebook -> cùng nhãn, cùng split.
# Trọng số ensemble được CHỌN TRÊN VAL, sau đó mới đo test.
# ======================================================================================
if not (RUN_BATCH and RUN_RF_BASELINE):
    print("Bỏ qua baseline RF (RUN_BATCH/RUN_RF_BASELINE=False)")
else:
    import time, numpy as np, pandas as pd, torch
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import f1_score, accuracy_score

    RF_CAP_PER_CLASS = 40_000
    VIEW, SEED = "A", SEEDS[0]

    data, ids, names = select_view(D, M, VIEW)
    (Xtr, ytr), (Xva, yva), (Xte, yte) = data["train"], data["val"], data["test"]
    K = len(names)
    rng = np.random.default_rng(SEED)
    sub = np.concatenate([rng.choice(np.flatnonzero(ytr == c), min(RF_CAP_PER_CLASS, int((ytr == c).sum())), replace=False)
                          for c in np.unique(ytr)])

    t0 = time.time()
    rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED).fit(Xtr[sub], ytr[sub])
    rf_s = time.time() - t0

    def rf_proba(X):
        P = np.zeros((len(X), K), np.float32); P[:, rf.classes_] = rf.predict_proba(X); return P

    def mha_proba(model, X, bs=4096):
        model.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(X), bs):
                xb = torch.from_numpy(np.ascontiguousarray(X[i:i + bs], dtype=np.int64)).to(device)
                out.append(torch.softmax(model(xb).float(), 1).cpu().numpy())
        return np.concatenate(out)

    ck = torch.load(RESULTS / f"best_mha_view{VIEW}_seed{SEED}.pt", map_location=device, weights_only=False)
    mha = new_model(ck["cfg"], K, np.asarray(ck["feat_max"]), device); mha.load_state_dict(ck["state_dict"])

    P = {"val": (mha_proba(mha, Xva), rf_proba(Xva), yva), "test": (mha_proba(mha, Xte), rf_proba(Xte), yte)}
    mf1 = lambda y, p: f1_score(y, p, labels=range(K), average="macro", zero_division=0)

    # chọn trọng số trên VAL
    grid = np.round(np.arange(0, 1.01, 0.1), 2)
    val_scores = {w: mf1(yva, (w * P["val"][0] + (1 - w) * P["val"][1]).argmax(1)) for w in grid}
    w_best = max(val_scores, key=val_scores.get)

    rows = []
    for name, w in (("MHA", 1.0), ("RF", 0.0), (f"Ensemble(w_MHA={w_best})", w_best)):
        pm, pr, y = P["test"]; pred = (w * pm + (1 - w) * pr).argmax(1)
        f = f1_score(y, pred, labels=range(K), average=None, zero_division=0)
        rows.append(dict(model=name, val_macro_f1=val_scores.get(w, mf1(yva, (w * P['val'][0] + (1 - w) * P['val'][1]).argmax(1))),
                         test_macro_f1=f.mean(), test_acc=accuracy_score(y, pred), **{f"f1_{n}": v for n, v in zip(names, f)}))
    res = pd.DataFrame(rows)
    res.to_csv(RESULTS / f"compare_models_view{VIEW}_seed{SEED}.csv", index=False)
    print(f"RF train {rf_s:.0f}s | w_MHA chọn trên val = {w_best}")
    print(res.round(4).T.to_string())

## XUẤT TRIỂN KHAI (B) - Kiểm tra checkpoint, xuất TorchScript, đóng gói model bundle
Đặt cell này ngay SAU cell "NẠP DỮ LIỆU ĐÃ XỬ LÝ" (cần D, M, new_model, select_view, predict).
Chạy sau cell A trong CÙNG phiên.

(bật bằng RUN_DEPLOY_EXPORT)

In [ ]:
if not RUN_DEPLOY_EXPORT:
    print("Bỏ qua XUẤT TRIỂN KHAI (B) (RUN_DEPLOY_EXPORT=False)")
else:
    import shutil, joblib
    import torch
    from sklearn.metrics import f1_score, accuracy_score

    EXPORT = WORK_DIR / "deploy_export"
    BUNDLE = EXPORT / "bundle_v1"
    BUNDLE.mkdir(parents=True, exist_ok=True)
    VERSION = "v1-batch-viewA-seed42"

    # 1) Tìm checkpoint: ưu tiên bản VỪA huấn luyện trong phiên này (results), sau đó mới tới bản tải lên /kaggle/input
    #    (checkpoint phải được huấn luyện trên ĐÚNG dữ liệu tiền xử lý hiện tại)
    cands = [RESULTS / "best_mha_viewA_seed42.pt"] + sorted(Path(os.environ.get("WORK_DIR", ".")).rglob("best_mha_viewA_seed42.pt"))
    CKPT = next((p for p in cands if p.exists()), None)
    assert CKPT is not None, "Không thấy best_mha_viewA_seed42.pt - tải file lên Kaggle (Add Input) rồi chạy lại"
    print("Checkpoint:", CKPT)
    ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)
    names = list(ckpt["classes"])
    ccfg = ckpt["cfg"]
    assert not ccfg.get("per_feature_norm"), "Cell này giả định per_feature_norm=False (đúng cấu hình đã train)"

    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = new_model(ccfg, len(names), np.asarray(ckpt["feat_max"], dtype=np.float32), dev)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()

    # 2) Đánh giá lại trên token test -> phải khớp Macro-F1 lúc train (đọc từ summary_batch.csv nếu có)
    data, ids, names_v = select_view(D, M, "A")
    Xte, yte = data["test"]
    assert names_v == names, f"Thứ tự lớp khác checkpoint: {names_v} vs {names}"
    pred = predict(model, Xte, dev)
    acc = accuracy_score(yte, pred)
    mf1 = f1_score(yte, pred, average="macro")
    wf1 = f1_score(yte, pred, average="weighted")
    ref = None
    if (RESULTS / "summary_batch.csv").exists():
        sb = pd.read_csv(RESULTS / "summary_batch.csv")
        sb = sb[(sb["view"] == "A") & (sb["seed"] == 42)]
        ref = None if sb.empty else sb.iloc[-1]
    if ref is not None:
        print(f"Test: acc={acc:.4f} macro-F1={mf1:.4f} weighted-F1={wf1:.4f}  "
              f"(lúc train: {ref['test_acc']:.4f} / {ref['test_macro_f1']:.4f} / {ref['test_weighted_f1']:.4f})")
        assert abs(mf1 - ref["test_macro_f1"]) < 0.01, "Lệch nhiều so với lúc train -> checkpoint KHÔNG khớp dữ liệu hiện tại"
    else:
        print(f"Test: acc={acc:.4f} macro-F1={mf1:.4f} weighted-F1={wf1:.4f}  "
              "(CẢNH BÁO: không có summary_batch.csv của phiên này để đối chiếu)")

    # 3) Kiểm tra pipeline đầy đủ: CSV thô -> tokenizer -> giống hệt token test
    tok_params = joblib.load(EXPORT / "tokenizer.joblib")
    raw = pd.read_csv(EXPORT / "test_viewA_raw.csv")
    X_from_raw = raw_to_tokens(tok_params, raw)          # tự dựng cờ inf/missing + impute median như lúc train
    assert (X_from_raw == Xte.astype(np.int64)).all(), "Token từ CSV thô khác token test"
    assert (raw["Label"].map({n: i for i, n in enumerate(names)}).to_numpy() == yte).all(), "Nhãn/thu tự dòng lệch"
    print("CSV thô -> tokenizer -> token: KHỚP 100%")

    # 4) Xuất TorchScript (trace trên CPU, float32) và so sánh với model gốc
    cpu_model = model.to("cpu").float().eval()
    example = torch.from_numpy(Xte[:64].astype(np.int64))
    with torch.no_grad():
        ts = torch.jit.trace(cpu_model, example, check_trace=False)
        for bs in (1, 7, 64, 513):
            xb = torch.from_numpy(Xte[1000:1000 + bs].astype(np.int64))
            d = (cpu_model(xb) - ts(xb)).abs().max().item()
            assert d < 1e-4, f"TorchScript lệch {d} ở batch {bs}"
    print("TorchScript khớp model gốc ở các batch 1/7/64/513")
    ts.save(str(BUNDLE / "model_ts.pt"))

    # 5) Các file còn lại của bundle
    (BUNDLE / "features.json").write_text(json.dumps(M["features"], ensure_ascii=False, indent=1), encoding="utf-8")
    (BUNDLE / "raw_features.json").write_text(                       # cột THÔ server cần nhận (cờ do tokenizer tự dựng)
        json.dumps(M.get("raw_features", M["features"]), ensure_ascii=False, indent=1), encoding="utf-8")
    (BUNDLE / "label_map.json").write_text(
        json.dumps({str(i): n for i, n in enumerate(names)}, ensure_ascii=False, indent=1), encoding="utf-8")
    shutil.copy(EXPORT / "tokenizer.joblib", BUNDLE / "tokenizer.joblib")
    versions = json.loads((EXPORT / "versions.json").read_text())
    versions["torch"] = torch.__version__
    meta_b = dict(version=VERSION, input="tokens_int64", num_bins=int(ckpt["num_bins"]),
                  cfg=ccfg, versions=versions, source_checkpoint=CKPT.name,
                  offline_metrics=dict(accuracy=round(acc, 4), macro_f1=round(mf1, 4), weighted_f1=round(wf1, 4)))
    (BUNDLE / "meta.json").write_text(json.dumps(meta_b, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

    zip_path = shutil.make_archive(str(WORK_DIR / "deploy_export"), "zip", EXPORT)
    print("Bundle:", sorted(p.name for p in BUNDLE.iterdir()))
    print("Đã nén ->", zip_path, "(tải file này về)")

## INCREMENTAL LEARNING - kịch bản A: class-incremental   (THAY cell chạy kịch bản A cũ bằng cell này)


In [ ]:
rows_A = []
if RUN_IL and RUN_A:
    for seed in SEEDS:
        rows, _ = run_class_incremental(D, M, cfg, METHODS, seed, device, RESULTS, run_joint=RUN_JOINT)
        rows_A += rows
    dfA = pd.DataFrame(rows_A)
    dfA.to_csv(RESULTS / "summary_A_all_seeds.csv", index=False)
    print(f"\n=== KỊCH BẢN A: trung bình trên {len(SEEDS)} seed ===")
    print(dfA.groupby("method")[["avg_macro_f1", "forgetting", "bwt", "final_overall_f1", "total_update_s"]]
              .agg(["mean", "std"]).round(4).to_string())

    # Thông số theo stage, gộp mọi seed -> mean (± std nếu nhiều seed)
    st_all = pd.concat([pd.read_csv(RESULTS / f"il_A_stage_metrics_seed{s}.csv") for s in SEEDS], ignore_index=True)
    st_all.to_csv(RESULTS / "il_A_stage_metrics_all_seeds.csv", index=False)
    cols = ["accuracy", "macro_f1", "weighted_f1", "macro_precision", "macro_recall", "update_s"]
    agg = st_all.groupby(["method", "stage"])[cols].agg(["mean", "std"]).round(4)
    agg.to_csv(RESULTS / "il_A_stage_metrics_summary.csv")
    print("\n=== Thông số theo stage (mean trên các seed) ===")
    for c in ("accuracy", "macro_f1", "weighted_f1"):
        print(f"\n{c}:")
        print(st_all.pivot_table(index="method", columns="stage", values=c, aggfunc="mean").round(4).to_string())
else:
    print("Bỏ qua kịch bản A")


## INCREMENTAL LEARNING - kịch bản B: drift theo ngày


In [ ]:
rows_B = []
if RUN_IL and RUN_B:
    for seed in SEEDS:
        rows, _ = run_domain_drift(D, M, cfg, METHODS, seed, device, RESULTS, run_joint=RUN_JOINT)
        rows_B += rows
    dfB = pd.DataFrame(rows_B)
    dfB.to_csv(RESULTS / "summary_B_all_seeds.csv", index=False)
    print(f"\n=== KỊCH BẢN B: trung bình trên {len(SEEDS)} seed ===")
    print(dfB.groupby("method")[["f1_day1", "f1_day2", "update_s"]].agg(["mean", "std"]).round(4).to_string())
else:
    print("Bỏ qua kịch bản B")

## ĐÓNG GÓI KẾT QUẢ


In [ ]:
for p in sorted(RESULTS.glob("*")):
    print(f"{p.name:<55} {p.stat().st_size / 1024:>10.1f} KB")
for name in ("meta.json", "feature_groups_report.json", "stats_class_day_split.csv"):
    if (PROCESSED / name).exists():
        shutil.copy(PROCESSED / name, RESULTS / name)
zip_path = shutil.make_archive(str(WORK_DIR / "ddos_results"), "zip", RESULTS)
print("\nĐã nén toàn bộ kết quả:", zip_path)

## COLAB 6 - Kết quả nằm ở đâu

In [ ]:
print("Kết quả (Drive):", WORK_DRIVE / "results")
print("File nén để tải về:", WORK_DRIVE / "ddos_results.zip")
print("Dữ liệu đã xử lý:", WORK_DRIVE / "processed", "| trung gian:", WORK_DRIVE / "interim")
from google.colab import drive
drive.flush_and_unmount()   # ghi hết bộ đệm lên Drive trước khi phiên kết thúc
print("Đã đồng bộ Drive.")